In [ ]:
# 1. Pobranie danych (API vs cache sterowane parametrem)

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import pandas as pd
import yfinance as yf


class DataIngestionError(RuntimeError):
    """Błąd pobierania / wczytywania danych (punkt 1)."""


@dataclass(frozen=True)
class DataIngestionConfig:
    """
    Konfiguracja pobrania danych rynkowych.

    Parametry
    ---------
    ticker : str
        Symbol instrumentu (np. "^GSPC", "AAPL", "BTC-USD").
        Wpływ: determinuje źródłową serię czasową (inne zachowanie rynku => inny sygnał/szum).

    start_date : str
        Data startowa w formacie YYYY-MM-DD.
        Wpływ: dłuższy zakres = więcej danych (zwykle mniejsza wariancja estymacji),
        ale potencjalnie więcej zmian reżimu (ryzyko biasu przez niestacjonarność).

    end_date : str | None
        Data końcowa w formacie YYYY-MM-DD lub None (domyślnie „do dziś” wg API).

    interval : str
        Interwał danych, np. "1d", "1h".
        Wpływ: krótszy interwał = więcej obserwacji (niższa wariancja), ale większy szum.

    data_dir : Path
        Katalog na cache danych (parquet/csv).

    data_source : str
        Sterowanie źródłem danych:
        - "api"   : wymuś pobranie z API i nadpisz cache
        - "cache" : wymuś wczytanie z cache (jeśli brak plików -> wyjątek)
        - "auto"  : jeśli cache istnieje -> wczytaj, inaczej pobierz z API
    """
    ticker: str = "^GSPC"
    start_date: str = "1900-01-01"
    end_date: str | None = None
    interval: str = "1d"

    data_dir: Path = Path("data/market")
    data_source: str = "auto"  # "api" | "cache" | "auto"


def _build_paths(cfg: DataIngestionConfig) -> tuple[Path, Path]:
    base_name = f"{cfg.ticker.replace('^','')}_{cfg.interval}"
    parquet_path = cfg.data_dir / f"{base_name}.parquet"
    csv_path = cfg.data_dir / f"{base_name}.csv"
    return parquet_path, csv_path


def _load_from_cache(parquet_path: Path, csv_path: Path) -> pd.DataFrame:
    # Preferuj parquet (format roboczy), ale obsłuż też csv
    if parquet_path.exists():
        print(f"[INFO] Wczytywanie danych z cache (parquet): {parquet_path}")
        try:
            df = pd.read_parquet(parquet_path)
        except Exception as e:
            raise DataIngestionError(f"Nie udało się wczytać parquet: {parquet_path}") from e

        # (opcjonalnie) odtwórz CSV do podglądu
        if not csv_path.exists():
            try:
                df.to_csv(csv_path, index=True)
                print(f"[INFO] (Podgląd) Odtworzono CSV: {csv_path}")
            except Exception as e:
                raise DataIngestionError(f"Nie udało się odtworzyć CSV: {csv_path}") from e

        return df

    if csv_path.exists():
        print(f"[INFO] Wczytywanie danych z cache (csv): {csv_path}")
        try:
            df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        except Exception as e:
            raise DataIngestionError(f"Nie udało się wczytać CSV: {csv_path}") from e

        # (opcjonalnie) odtwórz parquet do projektu
        try:
            df.to_parquet(parquet_path)
            print(f"[INFO] (Projekt) Odtworzono parquet: {parquet_path}")
        except Exception as e:
            raise DataIngestionError(f"Nie udało się odtworzyć parquet: {parquet_path}") from e

        return df

    raise DataIngestionError(
        "Brak cache danych (parquet/csv). Ustaw data_source='api' albo 'auto', aby pobrać dane."
    )


def _download_from_api(cfg: DataIngestionConfig) -> pd.DataFrame:
    print("[INFO] Pobieranie danych z API (yfinance)...")
    try:
        df = yf.download(
            tickers=cfg.ticker,
            start=cfg.start_date,
            end=cfg.end_date,
            interval=cfg.interval,
            auto_adjust=False,
            progress=False,
        )
    except Exception as e:
        raise DataIngestionError("Błąd podczas pobierania danych z yfinance.") from e

    if df.empty:
        raise DataIngestionError("Pobrane dane są puste — sprawdź ticker / zakres dat / interwał.")

    # Spłaszczenie MultiIndex (czasem yfinance zwraca MultiIndex)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]

    # Standaryzacja nazw kolumn
    df = df.rename(
        columns={
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume",
        }
    )

    # Podstawowe czyszczenie: usuń wiersze z brakami (na tym etapie OK)
    df = df.dropna()

    return df


def load_market_data(cfg: DataIngestionConfig) -> pd.DataFrame:
    allowed = {"api", "cache", "auto"}
    if cfg.data_source not in allowed:
        raise DataIngestionError(
            f"Nieprawidłowa wartość data_source={cfg.data_source!r}. Dozwolone: {sorted(allowed)}"
        )

    cfg.data_dir.mkdir(parents=True, exist_ok=True)
    parquet_path, csv_path = _build_paths(cfg)

    try:
        if cfg.data_source == "cache":
            df = _load_from_cache(parquet_path, csv_path)

        elif cfg.data_source == "api":
            df = _download_from_api(cfg)
            # zapis/overwrite cache
            df.to_parquet(parquet_path)
            df.to_csv(csv_path, index=True)
            print(f"[INFO] Zapisano cache: {parquet_path}")
            print(f"[INFO] (Podgląd) Zapisano CSV: {csv_path}")

        else:  # auto
            if parquet_path.exists() or csv_path.exists():
                df = _load_from_cache(parquet_path, csv_path)
            else:
                df = _download_from_api(cfg)
                df.to_parquet(parquet_path)
                df.to_csv(csv_path, index=True)
                print(f"[INFO] Zapisano cache: {parquet_path}")
                print(f"[INFO] (Podgląd) Zapisano CSV: {csv_path}")

    except Exception as e:
        print("[ERROR] Punkt 1 przerwany:", repr(e))
        raise

    return df


# =========================
# UŻYCIE (tu sterujesz źródłem)
# =========================
CFG = DataIngestionConfig(
    ticker="^GSPC",
    start_date="1900-01-01",
    end_date=None,
    interval="1d",
    data_dir=Path("data/market"),
    data_source="cache",  # <- "api" / "cache" / "auto"
)

df = load_market_data(CFG)

print(df.head())
print(df.tail())
print(df.info())

In [ ]:
# Metody pomocnicze do naprawy lub usunięcia danych

from pathlib import Path
import pandas as pd


def drop_rows_by_dates_from_csv(
    df_feat: pd.DataFrame,
    *,
    csv_path: str | Path,
    date_col: str,
    df_date_col: str | None = None,
    out_dir: str | Path,
    out_filename: str = "market_after_filtering.csv",
) -> pd.DataFrame:
    """
    Usuwa rekordy z df_feat na podstawie listy dat zapisanych w pliku CSV
    i zapisuje wynik do CSV w wskazanym katalogu.

    Parametry
    ----------
    df_feat : pd.DataFrame
        DataFrame z cechami (indeks DatetimeIndex lub kolumna z datą).

    csv_path : str | Path
        Ścieżka do pliku CSV zawierającego daty rekordów do usunięcia.

    date_col : str
        Nazwa kolumny w CSV zawierającej daty do usunięcia.

    df_date_col : str | None
        Jeśli None → używany jest DatetimeIndex df_feat.
        Jeśli podano → nazwa kolumny w df_feat zawierającej daty.

    out_dir : str | Path
        Katalog wyjściowy, gdzie zostanie zapisany CSV po filtrowaniu.

    out_filename : str
        Nazwa pliku CSV z danymi po filtrowaniu.

    Zwraca
    -------
    pd.DataFrame
        Nowy DataFrame z usuniętymi rekordami.

    Wyjątki
    --------
    FileNotFoundError
        Gdy plik CSV nie istnieje.

    ValueError
        Gdy CSV jest pusty, brak kolumny z datą,
        brak parsowalnych dat lub df_feat nie ma dat.
    """

    # ─────────────────────────────
    # 1) Walidacja pliku CSV
    # ─────────────────────────────
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"Plik CSV nie istnieje: {csv_path}")

    df_dates = pd.read_csv(csv_path)

    if df_dates.empty:
        raise ValueError(f"Plik CSV jest pusty: {csv_path}")

    if date_col not in df_dates.columns:
        raise ValueError(
            f"Brak kolumny '{date_col}' w CSV. "
            f"Dostępne kolumny: {list(df_dates.columns)}"
        )

    # ─────────────────────────────
    # 2) Parsowanie dat z CSV
    # ─────────────────────────────
    dates_raw = df_dates[date_col]

    dates = pd.to_datetime(dates_raw, errors="coerce")

    if dates.isna().all():
        raise ValueError(
            f"Nie udało się sparsować żadnej daty z kolumny '{date_col}' w CSV"
        )

    dates = dates.dropna().unique()

    # ─────────────────────────────
    # 3) Pobranie dat z df_feat
    # ─────────────────────────────
    df = df_feat.copy()

    if df_date_col is None:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError(
                "df_feat nie ma DatetimeIndex, a df_date_col=None. "
                "Podaj nazwę kolumny z datą w df_feat."
            )
        df_dates_feat = df.index
    else:
        if df_date_col not in df.columns:
            raise ValueError(
                f"Brak kolumny '{df_date_col}' w df_feat. "
                f"Dostępne kolumny: {list(df.columns)}"
            )
        df_dates_feat = pd.to_datetime(df[df_date_col], errors="coerce")

        if df_dates_feat.isna().all():
            raise ValueError(
                f"Kolumna '{df_date_col}' w df_feat nie zawiera parsowalnych dat"
            )

    # ─────────────────────────────
    # 4) Usuwanie rekordów
    # ─────────────────────────────
    mask_to_drop = df_dates_feat.isin(dates)

    n_drop = int(mask_to_drop.sum())

    if n_drop == 0:
        raise ValueError(
            "Nie znaleziono żadnych rekordów w df_feat "
            "odpowiadających datom z CSV"
        )

    df_clean = df.loc[~mask_to_drop].copy()

    print(
        f"[INFO] Usunięto {n_drop} rekordów na podstawie dat z pliku CSV "
        f"({csv_path})"
    )

    # ─────────────────────────────
    # 5) Zapis CSV po filtrowaniu
    # ─────────────────────────────
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / out_filename

    df_to_save = df_clean.copy()

    if df_date_col is None:
        if isinstance(df_to_save.index, pd.DatetimeIndex):
            df_to_save = df_to_save.reset_index().rename(columns={"index": "Date"})

    df_to_save.to_csv(out_path, index=False)

    print(f"[INFO] Zapisano dane po filtrowaniu do: {out_path}")

    return df_clean

# =========================
# 2a Usunięcie rekordów powyżej wskazanej daty
# =========================

from pathlib import Path
import pandas as pd


from pathlib import Path
import pandas as pd


def drop_rows_by_cutoff_date_to_csv(
    df: pd.DataFrame,
    *,
    date_col: str = "Date",
    cutoff_date: str = "1928-03-29",
    remove: str = "above",  # "above" albo "below"
    out_dir: str | Path = "data/validation",
) -> pd.DataFrame:
    """
    Usuwa rekordy względem daty granicznej.

    remove="above" -> usuń daty > cutoff_date (zostają <=)
    remove="below" -> usuń daty < cutoff_date (zostają >=)
    """

    if df is None or df.empty:
        raise ValueError("DataFrame jest pusty lub None.")

    if remove not in {"above", "below"}:
        raise ValueError("Parametr 'remove' musi być jednym z: {'above','below'}.")

    try:
        cutoff = pd.to_datetime(cutoff_date)
    except Exception as e:
        raise ValueError(f"Niepoprawny format daty: {cutoff_date}") from e

    df = df.copy()

    # --- pobierz serię dat (kolumna albo indeks) ---
    if date_col in df.columns:
        try:
            dates = pd.to_datetime(df[date_col])
        except Exception as e:
            raise ValueError(f"Nie można przekonwertować kolumny '{date_col}' na datetime.") from e
    elif df.index.name == date_col:
        if not isinstance(df.index, pd.DatetimeIndex):
            try:
                df.index = pd.to_datetime(df.index)
            except Exception as e:
                raise ValueError("Nie można przekonwertować indeksu na DatetimeIndex.") from e
        dates = df.index
    else:
        raise ValueError(f"'{date_col}' nie istnieje ani jako kolumna, ani jako nazwa indeksu.")

    # --- filtr ---
    if remove == "above":
        mask = dates <= cutoff   # zostają "poniżej / wcześniej"
    else:  # remove == "below"
        mask = dates >= cutoff   # zostają "powyżej / później"

    initial_rows = len(df)
    df_filtered = df.loc[mask].copy()

    if df_filtered.empty:
        raise ValueError("Po filtrowaniu DataFrame jest pusty.")

    # --- zapis ---
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    output_path = out_dir / f"market_after_filtering_{remove}_{cutoff.date()}.csv"

    # jeśli Date jest indeksem, zapisujemy go jawnie, żeby CSV był czytelny
    if df_filtered.index.name == date_col:
        df_filtered.to_csv(output_path, index=True)
    else:
        df_filtered.to_csv(output_path, index=False)

    removed_rows = initial_rows - len(df_filtered)
    print(f"[INFO] Usunięto {removed_rows} rekordów.")
    print(f"[INFO] Zakres dat po filtrowaniu: {df_filtered.index.min()} -> {df_filtered.index.max()}")
    print(f"[INFO] Zapisano do: {output_path}")

    return df_filtered

def _attach_ohlcv(
    df_features: pd.DataFrame,
    raw_df: pd.DataFrame,
    ohlcv_cols: list[str],
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Dokleja kolumny OHLCV (bez skalowania) z raw_df do df_features.
    Łączy po kolumnie date_col (preferowane) lub po DatetimeIndex.
    """
    if raw_df is None or not isinstance(raw_df, pd.DataFrame):
        raise EvaluationError(
            "Brak raw_df z OHLCV. "
            "Ustaw w notebooku np. raw_df / df_raw / data_raw i przekaż go do ewaluacji."
        )

    missing = [c for c in ohlcv_cols if c not in raw_df.columns]
    if missing:
        raise EvaluationError(f"raw_df nie ma kolumn OHLCV: {missing}")

    left = df_features.copy()

    # Upewnij się, że raw ma 'date' (albo index datetime)
    if date_col not in raw_df.columns:
        if isinstance(raw_df.index, pd.DatetimeIndex):
            raw = raw_df.copy()
            raw.insert(0, date_col, raw.index)
        else:
            raise EvaluationError(f"raw_df nie ma '{date_col}' ani DatetimeIndex – nie umiem zmatchować dat.")
    else:
        raw = raw_df.copy()

    # Upewnij się, że left ma 'date'
    if date_col not in left.columns:
        if isinstance(left.index, pd.DatetimeIndex):
            left.insert(0, date_col, left.index)
        else:
            raise EvaluationError(f"df_features nie ma '{date_col}' ani DatetimeIndex – nie umiem zmatchować dat.")

    # Join po dacie
    raw_small = raw[[date_col] + ohlcv_cols].drop_duplicates(subset=[date_col])
    out = left.merge(raw_small, on=date_col, how="left", validate="many_to_one")

    # Kontrola: czy coś się nie zmatchowało
    if out[ohlcv_cols].isna().any().any():
        # Nie przerywam od razu, ale to zazwyczaj błąd pipeline (inne kalendarze, strefy czasu, przesunięcia)
        nan_rows = int(out[ohlcv_cols].isna().any(axis=1).sum())
        raise EvaluationError(
            f"Nie udało się dopasować OHLCV dla {nan_rows} rekordów. "
            "Sprawdź, czy daty w raw_df i splitach są identyczne (ten sam timezone/format)."
        )

    return out


In [ ]:
# Punkt 2: Walidacja surowych danych

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence
import pandas as pd


# =========================
# Wyjątki / wyniki walidacji
# =========================

class ValidationError(ValueError):
    """Błąd walidacji danych surowych."""


@dataclass(frozen=True)
class ValidationIssue:
    validator: str
    check: str
    message: str
    n_bad: int = 0
    csv_path: Optional[Path] = None


# =========================
# Baza dla walidatorów
# =========================

class BaseRawValidator:
    """Interfejs dla pojedynczego walidatora surowych danych."""
    name: str = "BaseRawValidator"

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        raise NotImplementedError

    def _ensure_blacksheep_dir(self, blacksheep_dir: Optional[Path]) -> Path:
        if blacksheep_dir is None:
            raise ValidationError(
                "blacksheep_dir jest wymagany gdy saveBlackSheeps=True."
            )
        blacksheep_dir = Path(blacksheep_dir)
        blacksheep_dir.mkdir(parents=True, exist_ok=True)
        return blacksheep_dir

    def _save_blacksheeps(
        self,
        df_or_subset: pd.DataFrame,
        *,
        blacksheep_dir: Path,
        check: str,
    ) -> Path:
        # Uwaga: nie używam timestampów, żeby wyniki były deterministyczne przy powtarzalnych uruchomieniach.
        # Jeśli wolisz unikalne nazwy, łatwo dopisać suffix.
        safe_check = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in check)
        out_path = blacksheep_dir / f"{self.name}__{safe_check}.csv"
        df_or_subset.to_csv(out_path, index=True)
        return out_path


# =========================================
# Punkt 2: Walidacja surowych danych (cz. 1)
# 1) Indeks i oś czasu
# =========================================

class IndexAndTimeAxisValidator(BaseRawValidator):
    """
    Walidacje dla:
    1) Indeks i oś czasu

    Każdy podpunkt ma osobną metodę.
    """
    name = "IndexAndTimeAxisValidator"

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        # Kolejność ma znaczenie: najpierw typ indeksu, bo pozostałe checki na nim polegają. 
        issues += self.check_datetime_index(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)

        # Jeśli indeks nie jest DatetimeIndex, dalsze checki nie mają sensu.
        if issues and any(i.check == "check_datetime_index" for i in issues):
            raise ValidationError(self._format_issues(issues))

        issues += self.check_no_nat_in_index(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)
        issues += self.check_monotonic_increasing(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)
        issues += self.check_unique_index(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)
        issues += self.check_consistent_time_component_daily(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)
        issues += self.check_timezone_consistency(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)

        if issues:
            # W tym projekcie przyjmujemy twarde walidacje na raw.
            raise ValidationError(self._format_issues(issues))

        return issues

    # -------------------------
    # Podpunkt: DatetimeIndex - przetestowano
    # -------------------------

    def check_datetime_index(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        if isinstance(df.index, pd.DatetimeIndex):
            return []

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            # Nie da się jednoznacznie wskazać "złych rekordów" bez poprawnego indeksu czasu,
            # więc zapisujemy całe df jako blacksheeps.
            csv_path = self._save_blacksheeps(df, blacksheep_dir=out_dir, check="check_datetime_index")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_datetime_index",
                message=f"Index musi być pandas.DatetimeIndex, a jest: {type(df.index).__name__}.",
                n_bad=len(df),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: brak NaT w indeksie - przetestowano
    # -------------------------

    def check_no_nat_in_index(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        idx: pd.DatetimeIndex = df.index  # po check_datetime_index
        nat_mask = idx.isna() # NaT → brakująca data / czas

        if not nat_mask.any():
            return []

        bad_df = df.loc[nat_mask]
        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_no_nat_in_index")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_no_nat_in_index",
                message="Index zawiera NaT (brakujące daty/czasy) – to błąd na surowych danych.",
                n_bad=int(nat_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: monotonicznie rosnący - poprawiono - do ponownego testu
    # -------------------------

    def check_monotonic_increasing(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        idx = df.index
        if not isinstance(idx, pd.DatetimeIndex):
            raise ValidationError(
                f"[{self.name}.check_monotonic_increasing] Oczekiwano DatetimeIndex, "
                f"otrzymano: {type(idx).__name__}"
            )
    
        if idx.is_monotonic_increasing:
            return []
    
        # Rekordy "odpowiedzialne" za fail: miejsca, gdzie kolejny timestamp jest mniejszy od poprzedniego.
        # Zaznaczamy oba wiersze: poprzedni i bieżący.
        try:
            idx_values = idx.to_numpy()
            diffs = idx_values[1:] < idx_values[:-1]  # numpy bool array
            bad_pos_cur = (diffs.nonzero()[0] + 1)    # pozycje "bieżące"
            bad_pos = sorted(set(bad_pos_cur.tolist() + (bad_pos_cur - 1).tolist()))
            bad_df = df.iloc[bad_pos].copy()
        except Exception as e:
            print(f"[{self.name}.check_monotonic_increasing] Błąd przy wyznaczaniu bad_pos: {e!r}")
            raise
    
        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            try:
                out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
                # WAŻNE: zapisujemy do *out_dir* (a nie surowego blacksheep_dir),
                # żeby nie skończyć z None / nieutworzonym katalogiem.
                csv_path = self._save_blacksheeps(
                    bad_df,
                    blacksheep_dir=out_dir,
                    check="check_monotonic_increasing",
                )
            except Exception as e:
                # Jeśli zapis nie działa, chcemy to widzieć od razu (zamiast "cicho" gubić plik).
                print(f"[{self.name}.check_monotonic_increasing] Nie udało się zapisać blacksheepów: {e!r}")
                raise
    
        return [
            ValidationIssue(
                validator=self.name,
                check="check_monotonic_increasing",
                message="Index nie jest monotonicznie rosnący (oś czasu jest 'poszarpana').",
                n_bad=len(bad_df),
                csv_path=csv_path,
            )
        ]


    # -------------------------
    # Podpunkt: unikalny indeks (brak duplikatów) - przetestowano
    # -------------------------

    def check_unique_index(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        idx: pd.DatetimeIndex = df.index
        dup_mask = idx.duplicated(keep=False)

        if not dup_mask.any():
            return []

        bad_df = df.loc[dup_mask]
        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_unique_index")

        # Dodatkowy kontekst: ile unikalnych duplikowanych timestampów
        n_dup_groups = idx[dup_mask].unique().shape[0]

        return [
            ValidationIssue(
                validator=self.name,
                check="check_unique_index",
                message=f"Index zawiera duplikaty timestampów (liczba grup duplikatów: {n_dup_groups}).",
                n_bad=int(dup_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: "dziwne godziny" dla danych dziennych
    # -------------------------

    def check_consistent_time_component_daily(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        """
        Dla interwału 1d chcemy spójny komponent czasu w znacznikach (np. zawsze 00:00:00),
        bo mieszanie 00:00 i 16:00 potrafi psuć rolling/shift i joiny.

        Heurystyka:
        - bierzemy najczęstszy (mode) komponent czasu i wymagamy, by wszystkie były takie same.
        """
        idx: pd.DatetimeIndex = df.index

        # komponent czasu jako liczba ns od północy
        time_ns = (idx.hour.astype("int64") * 3600
                   + idx.minute.astype("int64") * 60
                   + idx.second.astype("int64")) * 1_000_000_000 + idx.nanosecond.astype("int64")

        # mode:
        vc = pd.Series(time_ns).value_counts(dropna=False)
        mode_time_ns = int(vc.index[0])
        bad_mask = time_ns != mode_time_ns

        if not bool(bad_mask.any()):
            return []

        bad_df = df.loc[bad_mask]

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_consistent_time_component_daily")

        # ładniejszy opis mode (HH:MM:SS)
        mode_sec = mode_time_ns // 1_000_000_000
        hh = mode_sec // 3600
        mm = (mode_sec % 3600) // 60
        ss = mode_sec % 60

        return [
            ValidationIssue(
                validator=self.name,
                check="check_consistent_time_component_daily",
                message=f"Niespójny komponent czasu w indeksie (dla 1d oczekuję stałej godziny; najczęstsza to {hh:02d}:{mm:02d}:{ss:02d}).",
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: spójność TZ (albo brak TZ, albo konsekwentnie jedna) - nie przetestowano - "Teoretycznie nieosiągalne dla DatetimeIndex, ale zostawiam jako guard."
    # -------------------------

    def check_timezone_consistency(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        idx: pd.DatetimeIndex = df.index

        # W pandas DatetimeIndex ma jedną tz albo None.
        # Ten check łapie przypadki pośrednie (np. index jako object -> już odpadliśmy),
        # oraz wymusza "albo brak TZ, albo jedna" (co i tak trzyma pandas).
        # Dodatkowo: jeżeli tz-aware, to zostawiamy (kontrakt mówi: albo brak TZ, albo konsekwentnie jedna).
        # Jeśli chcesz wymusić tz-naive, to tu zmienimy regułę.
        if idx.tz is None or idx.tz is not None:
            return []

        # Teoretycznie nieosiągalne dla DatetimeIndex, ale zostawiam jako guard.
        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(df, blacksheep_dir=out_dir, check="check_timezone_consistency")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_timezone_consistency",
                message="Niespójna strefa czasowa w indeksie (wymagana jedna TZ albo brak TZ).",
                n_bad=len(df),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Helper do formatowania
    # -------------------------

    def _format_issues(self, issues: Sequence[ValidationIssue]) -> str:
        lines = ["[RAW VALIDATION FAILED] 1) Indeks i oś czasu:"]
        for it in issues:
            extra = f" | blacksheeps: {it.csv_path}" if it.csv_path else ""
            lines.append(f"- {it.check}: {it.message} (n_bad={it.n_bad}){extra}")
        return "\n".join(lines)

# =========================================
# Punkt 2: Walidacja surowych danych (cz. 2)
# 2) Ciągłość dat (kalendarz sesji)
# =========================================

class SessionCalendarContinuityValidator(BaseRawValidator):
    """
    Walidacje dla:
    2) Ciągłość dat (kalendarz sesji)

    Kontrakt:
    - Poziom A (twardy): każda data w danych MUSI być dniem sesyjnym (NYSE dla ^GSPC).
      Jeśli nie -> ValidationError (+ opcjonalny zapis blacksheeps jako rekordy z błędnych dni).
    - Poziom B (miękki): brakujące sesje są dozwolone (nie imputujemy), ale logujemy WARNING
      (+ opcjonalny zapis listy brakujących sesji do CSV).
    """
    name = "SessionCalendarContinuityValidator"

    def __init__(self, calendar: str = "NYSE"):
        self.calendar = calendar

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        # Własny minimalny kontrakt wejścia (nie zależy od punktu 1 jako klasy)
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValidationError(
                f"{self.name} wymaga DatetimeIndex (otrzymano: {type(df.index).__name__})."
            )

        # A) Twardo: dni spoza kalendarza sesji
        issues += self.check_all_rows_are_session_days(
            df,
            saveBlackSheeps=saveBlackSheeps,
            blacksheep_dir=blacksheep_dir,
        )
        if issues and any(i.check == "check_all_rows_are_session_days" for i in issues):
            raise ValidationError(self._format_issues(issues))

        # B) Miękko: brakujące sesje (warning)
        issues += self.check_missing_sessions(
            df,
            saveBlackSheeps=saveBlackSheeps,
            blacksheep_dir=blacksheep_dir,
        )
        for w in [i for i in issues if i.check == "check_missing_sessions"]:
            extra = f" | blacksheeps: {w.csv_path}" if w.csv_path else ""
            print(f"[RAW VALIDATION WARNING] {self.name}.{w.check}: {w.message} (n_bad={w.n_bad}){extra}")

        return issues

    # -------------------------
    # Poziom A: daty w danych muszą być sesyjne - nie przetestowano
    # -------------------------

    def check_all_rows_are_session_days(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        sessions = self._get_sessions_for_range(df)

        # Porównujemy po "dniu" (normalize), bo indeks może mieć godzinę.
        data_days = df.index.normalize().unique()
        session_days = sessions.normalize().unique()

        bad_days = data_days.difference(session_days)
        if len(bad_days) == 0:
            return []

        # rekordy odpowiedzialne za fail: wszystkie wiersze z bad_days
        bad_mask = df.index.normalize().isin(bad_days)
        bad_df = df.loc[bad_mask].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(
                bad_df,
                blacksheep_dir=out_dir,
                check="check_all_rows_are_session_days",
            )

        sample = [d.strftime("%Y-%m-%d") for d in pd.to_datetime(bad_days[:5])]
        sample_txt = ", ".join(sample) + (" ..." if len(bad_days) > 5 else "")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_all_rows_are_session_days",
                message=(
                    f"Znaleziono rekordy dla dni nienależących do kalendarza sesji ({self.calendar}). "
                    f"Przykłady dni: {sample_txt}."
                ),
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Poziom B: brakujące sesje (warning) - nie przetestowano
    # -------------------------

    def check_missing_sessions(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        sessions = self._get_sessions_for_range(df)

        data_days = df.index.normalize().unique()
        session_days = sessions.normalize().unique()

        missing_days = session_days.difference(data_days)
        if len(missing_days) == 0:
            return []

        # zapiszemy listę braków (to nie są "rekordy df", więc robimy osobny DF)
        missing_df = pd.DataFrame({"missing_session_day": pd.to_datetime(missing_days).sort_values()})

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            # nadpisujemy _save_blacksheeps? nie, używamy go i zapisujemy z index=True (będzie 0..n-1)
            # bo BaseRawValidator i tak zapisuje index=True — to OK.
            csv_path = self._save_blacksheeps(
                missing_df,
                blacksheep_dir=out_dir,
                check="check_missing_sessions",
            )

        sample = [d.strftime("%Y-%m-%d") for d in pd.to_datetime(missing_days[:5])]
        sample_txt = ", ".join(sample) + (" ..." if len(missing_days) > 5 else "")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_missing_sessions",
                message=(
                    f"W danych brakuje niektórych dni sesyjnych ({self.calendar}). "
                    f"Nie imputujemy (zgodnie z kontraktem). Przykłady braków: {sample_txt}."
                ),
                n_bad=len(missing_days),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Backend kalendarza (exchange_calendars -> fallback pandas_market_calendars) - nie przetestowano
    # -------------------------

    def _get_sessions_for_range(self, df: pd.DataFrame) -> pd.DatetimeIndex:
        start = df.index.min().normalize()
        end = df.index.max().normalize()

        # 1) exchange_calendars
        try:
            import exchange_calendars as ecals  # type: ignore
            cal_code = "XNYS" if self.calendar.upper() == "NYSE" else self.calendar
            cal = ecals.get_calendar(cal_code)
            sessions = cal.sessions_in_range(start, end)
            if sessions.tz is not None:
                sessions = sessions.tz_convert(None)
            return pd.DatetimeIndex(sessions)
        except Exception:
            pass

        # 2) pandas_market_calendars
        try:
            import pandas_market_calendars as mcal  # type: ignore
            cal = mcal.get_calendar(self.calendar.upper())
            schedule = cal.schedule(start_date=start.date(), end_date=end.date())
            sessions = pd.DatetimeIndex(schedule.index)
            if sessions.tz is not None:
                sessions = sessions.tz_convert(None)
            return sessions
        except Exception as e:
            raise ValidationError(
                "Nie mogę załadować kalendarza sesji. "
                "Zainstaluj jedno z: exchange_calendars lub pandas_market_calendars.\n"
                f"Szczegóły: {repr(e)}"
            )

    def _format_issues(self, issues: Sequence[ValidationIssue]) -> str:
        lines = [f"[RAW VALIDATION FAILED] 2) Ciągłość dat (kalendarz sesji: {self.calendar}):"]
        for it in issues:
            extra = f" | blacksheeps: {it.csv_path}" if it.csv_path else ""
            lines.append(f"- {it.check}: {it.message} (n_bad={it.n_bad}){extra}")
        return "\n".join(lines)


# =========================================
# Punkt 2: Walidacja surowych danych (cz. 3)
# 3) Walidacja kolumn i typów
# =========================================

class ColumnsAndTypesValidator(BaseRawValidator):
    """
    Walidacje dla:
    3) Walidacja kolumn i typów

    Twarde zasady na RAW (fail fast):
    - Muszą istnieć kolumny: Open, High, Low, Close
    - Kolumny OHLC muszą być numeryczne i konwertowalne do float64
    - Brak NaN w OHLC

    Opcjonalnie (jeśli kolumny istnieją):
    - Adj Close i Volume nie są wymagane, ale jeśli są -> też walidujemy typ (i NaN dla Adj Close).
      (Volume: dopuszczamy NaN? na RAW zwykle nie; tu trzymam twardo: brak NaN jeśli istnieje.)
    """
    name = "ColumnsAndTypesValidator"

    def __init__(
        self,
        required_cols: Sequence[str] = ("open","high","low","close", "volume"),
        optional_cols: Sequence[str] = (),
        enforce_no_nan_optional: bool = True,
    ):
        self.required_cols = list(required_cols)
        self.optional_cols = list(optional_cols)
        self.enforce_no_nan_optional = bool(enforce_no_nan_optional)

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        issues += self.check_required_columns_present(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )

        # Jeśli brakuje required columns, dalsze checki nie mają sensu.
        if issues and any(i.check == "check_required_columns_present" for i in issues):
            raise ValidationError(self._format_issues(issues))

        issues += self.check_required_columns_numeric_castable(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )
        issues += self.check_no_nan_in_required(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )

        # Optional columns: walidujemy tylko jeśli istnieją
        issues += self.check_optional_columns_types(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )
        if self.enforce_no_nan_optional:
            issues += self.check_no_nan_in_optional(
                df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
            )

        if issues:
            raise ValidationError(self._format_issues(issues))

        return issues

    # -------------------------
    # Podpunkt: wymagane kolumny istnieją - przetestowano
    # -------------------------

    def check_required_columns_present(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        missing = [c for c in self.required_cols if c not in df.columns]
        if not missing:
            return []

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            # Brak kolumn -> nie ma "złych rekordów" per-row; zapisujemy snapshot całego df (nagłówki też)
            csv_path = self._save_blacksheeps(df, blacksheep_dir=out_dir, check="check_required_columns_present")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_required_columns_present",
                message=f"Brakuje wymaganych kolumn: {missing}. Dostępne kolumny: {list(df.columns)}.",
                n_bad=len(missing),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: wymagane kolumny numeryczne i castowalne do float64 - przetestowano
    # -------------------------

    def check_required_columns_numeric_castable(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        bad_cols: list[str] = []
        for c in self.required_cols:
            # najpierw szybki typ-check
            if pd.api.types.is_numeric_dtype(df[c]):
                continue
            # próbujemy twardej konwersji
            try:
                pd.to_numeric(df[c], errors="raise")
            except Exception:
                bad_cols.append(c)

        if not bad_cols:
            return []

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            # zapisujemy tylko problematyczne kolumny + index, żeby łatwiej debugować
            csv_path = self._save_blacksheeps(df[bad_cols].copy(), blacksheep_dir=out_dir, check="check_required_columns_numeric_castable")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_required_columns_numeric_castable",
                message=(
                    "Niektóre wymagane kolumny nie są numeryczne i/lub nie dają się bezpiecznie "
                    f"zrzutować do typu liczbowego: {bad_cols}."
                ),
                n_bad=len(bad_cols),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: brak NaN w wymaganych OHLC - przetestowano
    # -------------------------

    def check_no_nan_in_required(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        sub = df[self.required_cols]
        nan_mask = sub.isna().any(axis=1)
        if not bool(nan_mask.any()):
            return []

        bad_df = df.loc[nan_mask, self.required_cols].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_no_nan_in_required")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_no_nan_in_required",
                message="W wymaganych kolumnach OHLC wykryto NaN (to błąd na surowych danych).",
                n_bad=int(nan_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: optional columns (jeśli istnieją) — typy - nie przetestowano
    # -------------------------

    def check_optional_columns_types(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        present_optional = [c for c in self.optional_cols if c in df.columns]
        if not present_optional:
            return []

        bad_cols: list[str] = []
        for c in present_optional:
            if pd.api.types.is_numeric_dtype(df[c]):
                continue
            try:
                pd.to_numeric(df[c], errors="raise")
            except Exception:
                bad_cols.append(c)

        if not bad_cols:
            return []

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(df[bad_cols].copy(), blacksheep_dir=out_dir, check="check_optional_columns_types")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_optional_columns_types",
                message=f"Opcjonalne kolumny istnieją, ale mają niepoprawny typ/format liczbowy: {bad_cols}.",
                n_bad=len(bad_cols),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Podpunkt: optional columns (jeśli istnieją) — brak NaN - nie przetestowano
    # -------------------------

    def check_no_nan_in_optional(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        present_optional = [c for c in self.optional_cols if c in df.columns]
        if not present_optional:
            return []

        sub = df[present_optional]
        nan_mask = sub.isna().any(axis=1)
        if not bool(nan_mask.any()):
            return []

        bad_df = df.loc[nan_mask, present_optional].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_no_nan_in_optional")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_no_nan_in_optional",
                message=f"W opcjonalnych kolumnach {present_optional} wykryto NaN (ustawione jako błąd na RAW).",
                n_bad=int(nan_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Helper do formatowania
    # -------------------------

    def _format_issues(self, issues: Sequence[ValidationIssue]) -> str:
        lines = ["[RAW VALIDATION FAILED] 3) Walidacja kolumn i typów:"]
        for it in issues:
            extra = f" | blacksheeps: {it.csv_path}" if it.csv_path else ""
            lines.append(f"- {it.check}: {it.message} (n_bad={it.n_bad}){extra}")
        return "\n".join(lines)

# =========================================
# Punkt 2: Walidacja surowych danych (cz. 4)
# 4) Walidacja wartości (4.1, 4.2, 4.3)
# =========================================

class ValueSanityValidator(BaseRawValidator):
    """
    Walidacje dla:
    4) Walidacja wartości (tylko 4.1, 4.2, 4.3)

    4.1 (ERROR): OHLC muszą być > 0
    4.2 (ERROR): logika OHLC (High/Low vs Open/Close)
    4.3 (ERROR, gdy istnieje Volume):
        - O=H=L=C i Volume==0 -> ERROR
        - O=H=L=C i Volume>0 przez N kolejnych dni -> ERROR (N = flat_candle_min_run)
      Jeśli Volume nie istnieje -> ten punkt jest pomijany (bez ERROR / bez WARNING).
    """
    name = "ValueSanityValidator"

    def __init__(
        self,
        ohlc_cols: Sequence[str] = ("close", "high", "low", "open"),
        volume_col: str = "volume",
        flat_candle_min_run: int = 5,
    ):
        self.ohlc_cols = list(ohlc_cols)
        self.volume_col = volume_col

        if not isinstance(flat_candle_min_run, int) or flat_candle_min_run < 1:
            raise ValueError("flat_candle_min_run musi być int >= 1.")
        self.flat_candle_min_run = flat_candle_min_run

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        # 4.1
        issues += self.check_prices_positive(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )
        if issues and any(i.check == "check_prices_positive" for i in issues):
            raise ValidationError(self._format_issues(issues))

        # 4.2
        issues += self.check_ohlc_logic(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )
        if issues and any(i.check == "check_ohlc_logic" for i in issues):
            raise ValidationError(self._format_issues(issues))

        # 4.3 (tylko jeśli mamy Volume)
        issues += self.check_flat_candles_with_volume_policy(
            df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir
        )
        if issues and any(i.check == "check_flat_candles_with_volume_policy" for i in issues):
            raise ValidationError(self._format_issues(issues))

        return issues

    # -------------------------
    # 4.1: Dodatniość cen (OHLC > 0) - do przemyślenia - rzuca błędem tylko dla wymaganych kolumn w init - czy to błąd?
    # -------------------------
    def check_prices_positive(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        missing_cols = [c for c in self.ohlc_cols if c not in df.columns]
        if missing_cols:
            # Ten walidator zakłada, że punkt 3 wyłapał braki; tu robimy twardy guard.
            raise ValidationError(
                f"{self.name}.check_prices_positive wymaga kolumn {self.ohlc_cols}, brakuje: {missing_cols}."
            )

        sub = df[self.ohlc_cols]
        bad_mask = (sub <= 0).any(axis=1)

        if not bool(bad_mask.any()):
            return []

        bad_df = df.loc[bad_mask, self.ohlc_cols].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(
                bad_df, blacksheep_dir=out_dir, check="check_prices_positive"
            )

        return [
            ValidationIssue(
                validator=self.name,
                check="check_prices_positive",
                message="Wykryto OHLC <= 0 (ceny muszą być dodatnie na surowych danych).",
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # 4.2: Logika OHLC - poprawiono - do ponownego testu
    #   High >= max(Open, Close)
    #   Low  <= min(Open, Close)
    #   High >= Low
    # -------------------------
    def check_ohlc_logic(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        missing_cols = [c for c in self.ohlc_cols if c not in df.columns]
        if missing_cols:
            raise ValidationError(
                f"{self.name}.check_ohlc_logic wymaga kolumn {self.ohlc_cols}, brakuje: {missing_cols}."
            )

        # Mapowanie OHLC po nazwach (bez zależności od kolejności w self.ohlc_cols)
        colmap = {str(c).lower(): c for c in self.ohlc_cols}
        required = ("open", "high", "low", "close")
        missing_keys = [k for k in required if k not in colmap]
        if missing_keys:
            raise ValidationError(
                f"{self.name}.check_ohlc_logic wymaga, aby self.ohlc_cols zawierało nazwy: {list(required)} "
                f"(case-insensitive). Brakuje: {missing_keys}. Otrzymano: {self.ohlc_cols}."
            )

        o = df[colmap["open"]]
        h = df[colmap["high"]]
        l = df[colmap["low"]]
        c = df[colmap["close"]]

        bad_mask = (
            (h < l) |
            (h < o) |
            (h < c) |
            (l > o) |
            (l > c)
        )

        if not bool(bad_mask.any()):
            return []

        # zapisujemy pełny kontekst świecy
        bad_df = df.loc[bad_mask, self.ohlc_cols].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(
                bad_df, blacksheep_dir=out_dir, check="check_ohlc_logic"
            )

        return [
            ValidationIssue(
                validator=self.name,
                check="check_ohlc_logic",
                message="Wykryto niespójne OHLC (naruszona logika High/Low względem Open/Close).",
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]


    # -------------------------
    # 4.3: Flat candle policy (tylko jeśli istnieje Volume) - poprawiono - do ponownego testu
    #   - O=H=L=C i Volume==0 -> ERROR
    #   - O=H=L=C i Volume>0 przez N kolejnych dni -> ERROR
    # -------------------------
    def check_flat_candles_with_volume_policy(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        # Jeśli nie ma Volume -> pomijamy punkt 4.3
        if self.volume_col not in df.columns:
            return []

        missing_ohlc = [c for c in self.ohlc_cols if c not in df.columns]
        if missing_ohlc:
            raise ValidationError(
                f"{self.name}.check_flat_candles_with_volume_policy wymaga kolumn {self.ohlc_cols}, brakuje: {missing_ohlc}."
            )

        # Mapowanie OHLC po nazwach (bez zależności od kolejności w self.ohlc_cols)
        colmap = {str(c).lower(): c for c in self.ohlc_cols}
        required = ("open", "high", "low", "close")
        missing_keys = [k for k in required if k not in colmap]
        if missing_keys:
            raise ValidationError(
                f"{self.name}.check_flat_candles_with_volume_policy wymaga, aby self.ohlc_cols zawierało nazwy: {list(required)} "
                f"(case-insensitive). Brakuje: {missing_keys}. Otrzymano: {self.ohlc_cols}."
            )

        # Definicja flat candle
        o = df[colmap["open"]]
        h = df[colmap["high"]]
        l = df[colmap["low"]]
        c = df[colmap["close"]]
        v = df[self.volume_col]

        flat_mask = (o == h) & (h == l) & (l == c)

        # Reguła 1: flat + volume == 0
        zero_vol_mask = flat_mask & (v == 0)

        # Reguła 2: flat + volume > 0 przez N kolejnych dni
        pos_vol_flat = flat_mask & (v > 0)

        # Wykrywamy runy True o długości >= N
        run_bad_mask = self._mask_runs_ge_n(pos_vol_flat, n=self.flat_candle_min_run)

        bad_mask = zero_vol_mask | run_bad_mask

        if not bool(bad_mask.any()):
            return []

        # zapisujemy OHLC + Volume dla kontekstu
        cols_to_save = list(self.ohlc_cols) + [self.volume_col]
        bad_df = df.loc[bad_mask, cols_to_save].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(
                bad_df, blacksheep_dir=out_dir, check="check_flat_candles_with_volume_policy"
            )

        msg_parts = []
        if bool(zero_vol_mask.any()):
            msg_parts.append("flat candle z Volume==0")
        if bool(run_bad_mask.any()):
            msg_parts.append(f"flat candle z Volume>0 przez >= {self.flat_candle_min_run} kolejnych dni")
        msg = "Wykryto: " + " oraz ".join(msg_parts) + "."

        return [
            ValidationIssue(
                validator=self.name,
                check="check_flat_candles_with_volume_policy",
                message=msg,
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]


    # -------------------------
    # Helper: maska wszystkich elementów należących do runów True o długości >= n
    # -------------------------
    def _mask_runs_ge_n(self, mask: pd.Series, n: int) -> pd.Series:
        """
        Zwraca boolean maskę (o tym samym indexie), gdzie True oznacza,
        że dany element należy do runu True o długości >= n.
        Wymaga, żeby mask była Series o indexie df.
        """
        if mask.empty:
            return mask

        # Upewniamy się, że to bool Series
        s = mask.astype(bool)

        # Grupowanie runów: zmiana wartości tworzy nową grupę
        grp = s.ne(s.shift(fill_value=False)).cumsum()

        # Długości grup True
        true_run_lengths = s.groupby(grp).transform("sum")

        # Element jest zły, jeśli należy do True oraz długość runu True >= n
        return s & (true_run_lengths >= n)

    # -------------------------
    # Helper do formatowania
    # -------------------------
    def _format_issues(self, issues: Sequence[ValidationIssue]) -> str:
        lines = ["[RAW VALIDATION FAILED] 4) Walidacja wartości (4.1, 4.2, 4.3):"]
        for it in issues:
            extra = f" | blacksheeps: {it.csv_path}" if it.csv_path else ""
            lines.append(f"- {it.check}: {it.message} (n_bad={it.n_bad}){extra}")
        return "\n".join(lines)

# =========================
# Finalny runner (szkielet)
# =========================

class RawValidationRunner:
    """
    Docelowo: tu zbierzesz wszystkie walidatory (po jednym na umówioną walidację)
    i odpalisz je w jednym miejscu.
    """
    def __init__(self, validators: Sequence[BaseRawValidator]):
        self.validators = list(validators)

    def run(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> None:
        for v in self.validators:
            # Każdy walidator sam rzuca ValidationError jeśli coś nie gra.
            v.validate(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)


# ============================================================
# Wywołanie w kontekście "pierwszy_projekt_v2" (fragment użycia)
# ============================================================


BLACKSHEEP_DIR = Path("data/validation/blacksheeps")  # albo gdzie chcesz

runner = RawValidationRunner(
    validators=[
        IndexAndTimeAxisValidator(),
        SessionCalendarContinuityValidator(calendar="NYSE"),
        ColumnsAndTypesValidator(),
        ValueSanityValidator(flat_candle_min_run=5),
    ]
)

try:
     runner.run(df, saveBlackSheeps=True, blacksheep_dir=BLACKSHEEP_DIR)
     print("[INFO] Raw validation OK: 1) Indeks i oś czasu")
except ValidationError as e:
     # zgodnie z Twoim wymaganiem: wyraźnie drukujemy błąd (i/lub pozwalamy mu polecieć dalej)
     print(str(e))
     raise


In [ ]:
df = drop_rows_by_dates_from_csv(
    df,
    csv_path="data/validation/blacksheeps/ValueSanityValidator__check_flat_candles_with_volume_policy.csv",
    date_col="Date",
    out_dir="data/validation"
)

In [ ]:
# Usunięcie rekordów z 0 volumenem

#df = drop_rows_by_cutoff_date_to_csv(
#    df,
#    date_col="Date",
#    cutoff_date="1971-12-31",
#    remove="below",
#    out_dir="data/validation",
#)

In [ ]:
BLACKSHEEP_DIR = Path("data/validation/blacksheeps")  # albo gdzie chcesz

runner = RawValidationRunner(
    validators=[
        IndexAndTimeAxisValidator(),
        SessionCalendarContinuityValidator(calendar="NYSE"),
        ColumnsAndTypesValidator(),
        ValueSanityValidator(flat_candle_min_run=5),
    ]
)

try:
     runner.run(df, saveBlackSheeps=True, blacksheep_dir=BLACKSHEEP_DIR)
     print("[INFO] Raw validation OK: 1) Indeks i oś czasu")
except ValidationError as e:
     # zgodnie z Twoim wymaganiem: wyraźnie drukujemy błąd (i/lub pozwalamy mu polecieć dalej)
     print(str(e))
     raise

In [ ]:
# =========================================
# Punkt 2: Walidacja surowych danych (cz. 4)
# 4) Walidacja wartości (4.4)
# 4.4) Heurystyki: rynek vs bug (outliery / skoki)
# =========================================

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence
import pandas as pd


# =========================
# Wyjątki / wyniki walidacji
# =========================

class ValidationError(ValueError):
    """Błąd walidacji danych surowych."""


@dataclass(frozen=True)
class ValidationIssue:
    validator: str
    check: str
    message: str
    n_bad: int = 0
    csv_path: Optional[Path] = None


# =========================
# Baza dla walidatorów
# =========================

class BaseRawValidator:
    """Interfejs dla pojedynczego walidatora surowych danych."""
    name: str = "BaseRawValidator"

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        raise NotImplementedError

    def _ensure_blacksheep_dir(self, blacksheep_dir: Optional[Path]) -> Path:
        if blacksheep_dir is None:
            raise ValidationError(
                "blacksheep_dir jest wymagany gdy saveBlackSheeps=True."
            )
        blacksheep_dir = Path(blacksheep_dir)
        blacksheep_dir.mkdir(parents=True, exist_ok=True)
        return blacksheep_dir

    def _save_blacksheeps(
        self,
        df_or_subset: pd.DataFrame,
        *,
        blacksheep_dir: Path,
        check: str,
    ) -> Path:
        # Uwaga: nie używam timestampów, żeby wyniki były deterministyczne przy powtarzalnych uruchomieniach.
        # Jeśli wolisz unikalne nazwy, łatwo dopisać suffix.
        safe_check = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in check)
        out_path = blacksheep_dir / f"{self.name}__{safe_check}.csv"
        df_or_subset.to_csv(out_path, index=True)
        return out_path

class MarketMoveVsBugValidator(BaseRawValidator):
    """
    4.4) Heurystyki: rynek vs bug (outliery / skoki)

    Cel:
    - NIE usuwamy outlierów "bo duże"
    - Twardo wywalamy przypadki, które wyglądają jak bug danych:
      (samotny ekstrem, niespójne metryki, wolumen bez sensu, zbyt częste ekstremy)

    Każdy test ma osobną metodę.
    Rekordy odpowiedzialne za fail zapisywane do CSV jak w innych walidatorach.
    """
    name = "MarketMoveVsBugValidator"

    def __init__(
        self,
        *,
        open_col: str = "open",
        high_col: str = "high",
        low_col: str = "low",
        close_col: str = "close",
        volume_col: str = "volume",
        # progi "ekstremu" (heurystyki)
        abs_return_extreme: float = 0.25,      # 25%
        abs_gap_extreme: float = 0.20,         # 20%
        hl_ratio_extreme: float = 1.30,        # High/Low >= 1.30
        # spójność: duży ruch powinien mieć sensowny range
        min_hl_ratio_for_extreme_move: float = 1.05,  # jeśli return/gap ekstremalny, oczekujemy High/Low >= 1.05
        # wolumen (jeśli jest)
        vol_rolling_window: int = 50,
        min_vol_ratio_for_extreme_move: float = 0.10,  # ekstremalny ruch z vol < 10% mediany rolling -> podejrzane
        max_vol_ratio_without_move: float = 50.0,      # kosmiczny wolumen bez ruchu -> podejrzane
        no_move_abs_return: float = 0.005,             # ~0.5% "brak ruchu"
        # częstość ekstremów: jeśli zbyt często -> raczej problem danych
        max_extreme_fraction: float = 0.01,            # 1% rekordów jako ekstremy -> alarm (dla indeksu to dużo)
        # które heurystyki uruchamiamy (None = wszystkie)
        validation_methods: Optional[Sequence[str]] = None,
    ):
        self.open_col = open_col
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col
        self.volume_col = volume_col

        self.abs_return_extreme = float(abs_return_extreme)
        self.abs_gap_extreme = float(abs_gap_extreme)
        self.hl_ratio_extreme = float(hl_ratio_extreme)
        self.min_hl_ratio_for_extreme_move = float(min_hl_ratio_for_extreme_move)

        self.vol_rolling_window = int(vol_rolling_window)
        self.min_vol_ratio_for_extreme_move = float(min_vol_ratio_for_extreme_move)
        self.max_vol_ratio_without_move = float(max_vol_ratio_without_move)
        self.no_move_abs_return = float(no_move_abs_return)

        self.max_extreme_fraction = float(max_extreme_fraction)
        # Lista metod walidacji do uruchomienia (po nazwach metod)
        _allowed_methods = ['check_lonely_extremes', 'check_extreme_move_without_range_support', 'check_volume_sanity_around_extremes', 'check_extremes_frequency']
        if validation_methods is None:
            self.validation_methods = list(_allowed_methods)
        else:
            self.validation_methods = list(validation_methods)
            unknown = [m for m in self.validation_methods if m not in _allowed_methods]
            if unknown:
                raise ValueError(
                    "Nieznane validation_methods: "
                    f"{unknown}. Dozwolone: {_allowed_methods}"
                )

        # Stabilny porządek: jeśli user podał swoje metody, zostawiamy ich kolejność.
        # Jeśli None, lecimy w domyślnej kolejności z _allowed_methods.


        if self.vol_rolling_window < 5:
            raise ValueError("vol_rolling_window musi być >= 5.")
        if not (0 < self.max_extreme_fraction < 1):
            raise ValueError("max_extreme_fraction musi być w (0, 1).")

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        # Guard: wymagane kolumny (punkt 3 powinien to zapewnić, ale robimy twardy guard)
        missing = [c for c in (self.open_col, self.high_col, self.low_col, self.close_col) if c not in df.columns]
        if missing:
            raise ValidationError(
                f"{self.name} wymaga kolumn {missing} (brakuje w df.columns). "
                "Upewnij się, że ColumnsAndTypesValidator oraz mapowanie nazw kolumn jest spójne."
            )


        # 4.4.x: uruchamiamy tylko wybrane heurystyki (kolejność jak w self.validation_methods)
        _method_map = {
            "check_lonely_extremes": self.check_lonely_extremes,
            "check_extreme_move_without_range_support": self.check_extreme_move_without_range_support,
            "check_volume_sanity_around_extremes": self.check_volume_sanity_around_extremes,
            "check_extremes_frequency": self.check_extremes_frequency,
        }

        # Tylko te checki są "fatal" (stop pipeline)
        _fatal_methods = {"check_extremes_frequency"}
        
        for method_name in self.validation_methods:
            fn = _method_map.get(method_name)
            if fn is None:
                # Teoretycznie nieosiągalne (walidowane w __init__), ale wolimy twardy guard.
                raise ValidationError(
                    f"{self.name}: metoda '{method_name}' nie jest dostępna w _method_map."
                )
        
            new_issues = fn(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)
            issues += new_issues
        
            if new_issues:
                if method_name in _fatal_methods:
                    # ERROR → stop pipeline
                    raise ValidationError(self._format_issues(issues))
        
                # WARN → tylko informacja + ewentualny CSV już zapisany w fn(...)
                # (zgodnie z Twoim wymaganiem: drukujemy wyraźnie podejrzane przypadki)
                print(
                    f"[WARN] {self.name}: {method_name} wykrył podejrzane przypadki "
                    f"(n_issues={len(new_issues)}). Pipeline idzie dalej."
                )


        return issues


    # -------------------------
    # 4.4 helpers: metryki i maski ekstremów
    # -------------------------

    def _compute_basic_metrics(self, df: pd.DataFrame) -> pd.DataFrame:
        o = df[self.open_col].astype("float64")
        h = df[self.high_col].astype("float64")
        l = df[self.low_col].astype("float64")
        c = df[self.close_col].astype("float64")

        prev_c = c.shift(1)
        ret = c / prev_c - 1.0
        gap = o / prev_c - 1.0
        hl_ratio = h / l

        out = pd.DataFrame(
            {
                "ret": ret,
                "gap": gap,
                "hl_ratio": hl_ratio,
                "prev_close": prev_c,
            },
            index=df.index,
        )
        return out

    def _extreme_mask(self, m: pd.DataFrame) -> pd.Series:
        # Pierwszy wiersz ma NaN w ret/gap -> nie flagujemy go
        return (
            (m["ret"].abs() >= self.abs_return_extreme) |
            (m["gap"].abs() >= self.abs_gap_extreme) |
            (m["hl_ratio"] >= self.hl_ratio_extreme)
        ).fillna(False)

    # -------------------------
    # 4.4.1: samotne ekstremy (brak sąsiadów)
    # -------------------------
    def check_lonely_extremes(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        m = self._compute_basic_metrics(df)
        extreme = self._extreme_mask(m)

        # "Samotny" = ekstremalny dziś, ale brak ekstremu w dniu -1 i +1
        neigh = extreme.shift(1, fill_value=False) | extreme.shift(-1, fill_value=False)
        lonely = extreme & ~neigh

        if not bool(lonely.any()):
            return []

        # Kontekst: zapisujemy też sąsiadów, żeby łatwiej ocenić
        pos = lonely.to_numpy().nonzero()[0]
        ctx_pos = sorted(set(pos.tolist() + (pos - 1).tolist() + (pos + 1).tolist()))
        ctx_pos = [p for p in ctx_pos if 0 <= p < len(df)]
        bad_df = df.iloc[ctx_pos].copy()

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_lonely_extremes")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_lonely_extremes",
                message=(
                    "Wykryto 'samotne' ekstremy (duży skok bez podobnych sygnałów w dniu -1 i +1). "
                    "To często oznacza bug danych / zły timestamp."
                ),
                n_bad=int(lonely.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # 4.4.2: duży return/gap bez wsparcia w zakresie świecy (range)
    # -------------------------
    def check_extreme_move_without_range_support(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        m = self._compute_basic_metrics(df)

        extreme_move = (
            (m["ret"].abs() >= self.abs_return_extreme) |
            (m["gap"].abs() >= self.abs_gap_extreme)
        ).fillna(False)

        # Jeśli ruch jest ekstremalny, oczekujemy że świeca ma sensowny zakres (High/Low)
        weak_range = (m["hl_ratio"] < self.min_hl_ratio_for_extreme_move).fillna(False)

        bad_mask = extreme_move & weak_range
        if not bool(bad_mask.any()):
            return []

        cols_to_save = [self.open_col, self.high_col, self.low_col, self.close_col]
        # dorzucamy metryki jako kontekst (łatwiej debugować)
        ctx = df.loc[bad_mask, cols_to_save].copy()
        ctx["ret"] = m.loc[bad_mask, "ret"]
        ctx["gap"] = m.loc[bad_mask, "gap"]
        ctx["hl_ratio"] = m.loc[bad_mask, "hl_ratio"]

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(ctx, blacksheep_dir=out_dir, check="check_extreme_move_without_range_support")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_extreme_move_without_range_support",
                message=(
                    "Wykryto ekstremalny return/gap bez wsparcia w zakresie świecy (High/Low zbyt małe). "
                    "To wygląda na błąd danych (np. zły Close/Open albo zły dzień)."
                ),
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # 4.4.3: wolumen a ekstremy / wolumen bez ruchu (jeśli istnieje)
    # -------------------------
    def check_volume_sanity_around_extremes(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        if self.volume_col not in df.columns:
            return []

        v = pd.to_numeric(df[self.volume_col], errors="coerce").astype("float64")
        if v.isna().any():
            # NaN w volume powinien zostać wyłapany wcześniej, ale tu fail-fast (bo metryki niepewne)
            raise ValidationError(f"{self.name} wykrył NaN w kolumnie {self.volume_col}; napraw RAW lub walidator 3).")

        m = self._compute_basic_metrics(df)
        extreme = self._extreme_mask(m)

        # rolling median wolumenu
        vol_med = v.rolling(self.vol_rolling_window, min_periods=max(5, self.vol_rolling_window // 5)).median()
        vol_ratio = (v / vol_med).replace([pd.NA, pd.NaT, float("inf")], pd.NA)

        # A) ekstremalny ruch + wolumen == 0 lub bardzo niski vs rolling median
        low_vol_on_extreme = extreme & ((v == 0) | (vol_ratio < self.min_vol_ratio_for_extreme_move)).fillna(False)

        # B) kosmiczny wolumen bez ruchu ceny (często bug / złe scalanie)
        no_move = (m["ret"].abs() <= self.no_move_abs_return).fillna(False)
        huge_vol_no_move = no_move & (vol_ratio > self.max_vol_ratio_without_move).fillna(False)

        bad_mask = low_vol_on_extreme | huge_vol_no_move
        if not bool(bad_mask.any()):
            return []

        cols_to_save = [self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col]
        bad_df = df.loc[bad_mask, cols_to_save].copy()
        bad_df["ret"] = m.loc[bad_mask, "ret"]
        bad_df["gap"] = m.loc[bad_mask, "gap"]
        bad_df["hl_ratio"] = m.loc[bad_mask, "hl_ratio"]
        bad_df["vol_med"] = vol_med.loc[bad_mask]
        bad_df["vol_ratio"] = vol_ratio.loc[bad_mask]

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_volume_sanity_around_extremes")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_volume_sanity_around_extremes",
                message=(
                    "Wykryto podejrzaną relację wolumenu do ruchu ceny: "
                    "(A) ekstremalny ruch z zerowym/bardzo niskim wolumenem lub "
                    "(B) kosmiczny wolumen bez ruchu ceny."
                ),
                n_bad=int(bad_mask.sum()),
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # 4.4.4: częstość ekstremów (jeśli jest za duża -> raczej problem danych)
    # -------------------------
    def check_extremes_frequency(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool,
        blacksheep_dir: Optional[Path],
    ) -> list[ValidationIssue]:
        m = self._compute_basic_metrics(df)
        extreme = self._extreme_mask(m)

        n = len(df)
        n_ext = int(extreme.sum())
        frac = (n_ext / n) if n > 0 else 0.0

        if frac <= self.max_extreme_fraction:
            return []

        # rekordy odpowiedzialne: wszystkie ekstremy + podstawowy kontekst
        cols_to_save = [self.open_col, self.high_col, self.low_col, self.close_col]
        bad_df = df.loc[extreme, cols_to_save].copy()
        bad_df["ret"] = m.loc[extreme, "ret"]
        bad_df["gap"] = m.loc[extreme, "gap"]
        bad_df["hl_ratio"] = m.loc[extreme, "hl_ratio"]

        csv_path: Optional[Path] = None
        if saveBlackSheeps:
            out_dir = self._ensure_blacksheep_dir(blacksheep_dir)
            csv_path = self._save_blacksheeps(bad_df, blacksheep_dir=out_dir, check="check_extremes_frequency")

        return [
            ValidationIssue(
                validator=self.name,
                check="check_extremes_frequency",
                message=(
                    f"Zbyt wysoka częstość ekstremów: {frac:.2%} (n_ext={n_ext}, n={n}). "
                    f"To zwykle oznacza problem danych (skala, split, timestamp, merge/resample). "
                    f"Próg max_extreme_fraction={self.max_extreme_fraction:.2%}."
                ),
                n_bad=n_ext,
                csv_path=csv_path,
            )
        ]

    # -------------------------
    # Helper do formatowania
    # -------------------------
    def _format_issues(self, issues: Sequence[ValidationIssue]) -> str:
        lines = ["[RAW VALIDATION FAILED] 4) Walidacja wartości (4.4): rynek vs bug:"]
        for it in issues:
            extra = f" | blacksheeps: {it.csv_path}" if it.csv_path else ""
            lines.append(f"- {it.check}: {it.message} (n_bad={it.n_bad}){extra}")
        return "\n".join(lines)

# =========================
# Finalny runner (szkielet)
# =========================

class RawValidationRunner:
    """
    Docelowo: tu zbierzesz wszystkie walidatory (po jednym na umówioną walidację)
    i odpalisz je w jednym miejscu.
    """
    def __init__(self, validators: Sequence[BaseRawValidator]):
        self.validators = list(validators)

    def run(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> None:
        for v in self.validators:
            # Każdy walidator sam rzuca ValidationError jeśli coś nie gra.
            v.validate(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)

# ============================================================
# Wywołanie w kontekście "pierwszy_projekt_v2" (fragment użycia)
# ============================================================


BLACKSHEEP_DIR = Path("data/validation/blacksheeps")  # albo gdzie chcesz

runner = RawValidationRunner(
    validators=[
        MarketMoveVsBugValidator(
            # kolumny OHLCV
            open_col="open",
            high_col="high",
            low_col="low",
            close_col="close",
            volume_col="volume",

            # progi "ekstremu" (heurystyki)
            abs_return_extreme=0.25,      # 10%
            abs_gap_extreme=0.20,         # 20%
            hl_ratio_extreme=1.3,         # High/Low vs body (wg Twojej definicji)

            # spójność: duży ruch powinien mieć sensowny range
            min_hl_ratio_for_extreme_move=1.05,

            # wolumen (jeśli jest)
            vol_rolling_window=50,
            min_vol_ratio_for_extreme_move=0.10,
            max_vol_ratio_without_move=50.0,
            no_move_abs_return=0.005,

            # częstość ekstremów: jeśli zbyt często -> raczej problem danych
            max_extreme_fraction=0.01,

            # które heurystyki uruchamiamy (None = wszystkie)
            validation_methods=[
                "check_lonely_extremes",
                "check_extreme_move_without_range_support",
                "check_volume_sanity_around_extremes",
                "check_extremes_frequency",
            ],
        ),
    ]
)

try:
     runner.run(df, saveBlackSheeps=True, blacksheep_dir=BLACKSHEEP_DIR)
     print("[INFO] Raw validation OK: 1) Indeks i oś czasu")
except ValidationError as e:
     # zgodnie z Twoim wymaganiem: wyraźnie drukujemy błąd (i/lub pozwalamy mu polecieć dalej)
     print(str(e))
     raise

# ============================================================
# INTERPRETACJA PARAMETRÓW HEURYSTYK (Market vs Bug)
# ============================================================
#
# Parametr: abs_return_extreme = 0.10
# ------------------------------------------------------------
# Wzór:
#     abs_return = |close_t - close_{t-1}| / close_{t-1}
#
# Przykład 1:
#     close_{t-1} = 1000
#     close_t     = 1105
#     abs_return  = 105 / 1000 = 0.105 = 10.5%
#     10.5% >= 10%  → EKSTREMUM (flag)
#
# Przykład 2:
#     close_{t-1} = 1000
#     close_t     = 1040
#     abs_return  = 40 / 1000 = 0.04 = 4%
#     4% < 10% → OK
#
# Interpretacja:
#     Wykrywa bardzo duże dzienne ruchy ceny zamknięcia.
#     Chroni przed błędami typu: split bez adjust, zła waluta, scale ×100.
#
# ------------------------------------------------------------
#
# Parametr: abs_gap_extreme = 0.20
# ------------------------------------------------------------
# Wzór:
#     abs_gap = |open_t - close_{t-1}| / close_{t-1}
#
# Przykład 1:
#     close_{t-1} = 1000
#     open_t      = 1250
#     abs_gap     = 250 / 1000 = 0.25 = 25%
#     25% >= 20% → EKSTREMALNY GAP (flag)
#
# Przykład 2:
#     close_{t-1} = 1000
#     open_t      = 1080
#     abs_gap     = 80 / 1000 = 0.08 = 8%
#     8% < 20% → OK
#
# Interpretacja:
#     Wykrywa nienaturalnie duże luki między sesjami.
#     Typowe źródła błędu:
#         - split bez adjust
#         - zmiana waluty
#         - rollover kontraktu futures
#         - pomieszane tickery
#
# ------------------------------------------------------------
#
# Parametr: hl_ratio_extreme = 2.0
# ------------------------------------------------------------
# Wzór:
#     hl_ratio = (high - low) / |close - open|
#
# Przykład 1:
#     high  = 120
#     low   = 80
#     open  = 100
#     close = 102
#
#     range = 40
#     body  = 2
#     hl_ratio = 40 / 2 = 20
#     20 >= 2 → EKSTREMUM (flag)
#
# Przykład 2:
#     high  = 110
#     low   = 95
#     open  = 100
#     close = 108
#
#     range = 15
#     body  = 8
#     hl_ratio = 15 / 8 = 1.87
#     1.87 < 2 → OK
#
# Interpretacja:
#     Wykrywa świece z ogromnym zakresem przy małym body.
#     Często oznaka:
#         - błędnego high/low
#         - pojedynczego ticka odstającego
#
# ------------------------------------------------------------
#
# Parametr: max_extreme_fraction = 0.01
# ------------------------------------------------------------
# Wzór:
#     extreme_fraction = (# ekstremów) / N
#
# Przykład 1:
#     20 ekstremów w 1000 dni
#     20 / 1000 = 0.02 = 2%
#     2% >= 1% → PROBLEM SYSTEMOWY (flag)
#
# Przykład 2:
#     5 ekstremów w 1000 dni
#     5 / 1000 = 0.005 = 0.5%
#     0.5% < 1% → OK
#
# Interpretacja:
#     Jeśli ekstremów jest za dużo → to nie outlier,
#     tylko problem strukturalny danych.
#
#     Typowe przypadki:
#         - zmiana waluty
#         - brak adjust przy splitach
#         - złe mapowanie API
#         - błędna agregacja danych
#
# ============================================================
# Cel walidacji:
#     Odróżnić:
#         REALNY RUCH RYNKOWY
#     od:
#         BŁĘDU DANYCH (bug systemowy)
#
#     Chroni pipeline ML przed:
#         - data leakage
#         - zaburzeniem rozkładu cech
#         - eksplozją gradientów
#         - dominacją outlierów w skalowaniu
# ============================================================



In [ ]:
# 3. EDA – S&P 500 daily data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
%matplotlib inline
# %config InlineBackend.figure_format = 'retina'   # jeśli używasz Jupyter

# =============================================================================
# Przygotowanie pomocniczych serii (już na etapie EDA – często robimy to tutaj)
# =============================================================================
df["return"]      = df["close"].pct_change()
df["log_return"]  = np.log(df["close"] / df["close"].shift(1))
df["range"]       = (df["high"] - df["low"]) / df["close"].shift(1)   # daily range %
df["body"]        = (df["close"] - df["open"]) / df["open"]
df["upper_wick"]  = (df["high"] - df[["open","close"]].max(axis=1)) / df["close"]
df["lower_wick"]  = (df[["open","close"]].min(axis=1) - df["low"]) / df["close"]

# =============================================================================
# 3.1 Podstawowe spojrzenie
# =============================================================================
print("\n=== Podstawowe informacje ===")
print(df.index.min().date(), "–", df.index.max().date())
print(f"Liczba rekordów: {len(df):,d}")
print(f"Średni dzienny zwrot: {df['return'].mean():.4%}")
print(f"Średni dzienny |log return|: {df['log_return'].abs().mean():.4%}")
print("\nStatystyki kluczowych kolumn:")
print(df[["open","high","low","close","volume","return","log_return"]].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).round(4))

# =============================================================================
# 3.2 Wizualizacja ceny i zwrotów
# =============================================================================
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                        gridspec_kw={'height_ratios': [3, 1.2, 1.2]})

# Cena (log scale!)
axs[0].plot(df["close"], color="navy", lw=1.1, label="Close")
axs[0].set_yscale("log")
axs[0].set_title(f"{CFG.ticker} – Cena zamknięcia (skala logarytmiczna)")
axs[0].legend(loc="upper left")

# Zwroty dzienne
axs[1].plot(df["return"], color="teal", lw=0.8, label="Daily return")
axs[1].axhline(0, color="black", lw=0.6, alpha=0.5)
axs[1].set_title("Dzienny prosty zwrot")
axs[1].legend(loc="upper left")

# Logarytmiczne zwroty (częściej używane w modelach)
axs[2].plot(df["log_return"], color="darkorange", lw=0.8, label="Daily log return")
axs[2].axhline(0, color="black", lw=0.6, alpha=0.5)
axs[2].set_title("Dzienny logarytmiczny zwrot")
axs[2].legend(loc="upper left")

plt.tight_layout()
plt.show()

# =============================================================================
# 3.3 Rozkłady – porównanie normalnego + fat tails
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(df["log_return"].dropna(), bins=300, kde=True, stat="density",
             color="darkorange", alpha=0.4, label="log returns")

# Dla porównania – normalny rozkład o takiej samej średniej i wariancji
x = np.linspace(df["log_return"].min(), df["log_return"].max(), 500)
normal_pdf = (1 / (df["log_return"].std() * np.sqrt(2*np.pi))) * \
             np.exp( - (x - df["log_return"].mean())**2 / (2 * df["log_return"].std()**2) )
ax.plot(x, normal_pdf, "k--", lw=1.4, label="Normalny (dopasowany)")

ax.set_title("Rozkład dziennych logarytmicznych zwrotów + porównanie z normalnym")
ax.set_xlabel("log return")
ax.set_xlim([-0.15, 0.15])   # zoom na centralną część – ogony widać dalej
ax.legend()
plt.show()

# =============================================================================
# 3.4 Autokorelacja zwrotów i kwadratów zwrotów (klucz do volatility clustering)
# =============================================================================
fig, axs = plt.subplots(2, 1, figsize=(12, 7))

plot_acf(df["log_return"].dropna(), lags=60, ax=axs[0], title="ACF – log returns")
plot_pacf(df["log_return"].dropna(), lags=60, ax=axs[1], title="PACF – log returns", method="ywm")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(df["log_return"].dropna()**2, lags=100, ax=ax,
         title="ACF – (log returns)²   ← volatility clustering")
ax.set_ylim(bottom=-0.05)
plt.show()

# =============================================================================
# 3.5 Największe ruchy w historii (pomaga zrozumieć ekstremalne przypadki)
# =============================================================================
print("\n=== 15 największych dziennych spadków ===")
print(df["return"].nsmallest(15).round(4) * 100)

print("\n=== 15 największych dziennych wzrostów ===")
print(df["return"].nlargest(15).round(4) * 100)

# 3.6 Heatmapa braków w czasie – dla kolumn OHLC(V)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --------------------------------------------------------------------
# Przygotowanie macierzy brakujących wartości (True = brak)
# --------------------------------------------------------------------
missing_matrix = df[["open", "high", "low", "close", "adj_close", "volume"]].isna()

# Jeśli adj_close jest identyczne z close → często yfinance to robi → możemy je pominąć w wizualizacji
if (df["close"] == df["adj_close"]).all():
    missing_matrix = missing_matrix.drop(columns=["adj_close"])
    print("[INFO] adj_close == close w całym zbiorze → pominięto w heatmapie")

# --------------------------------------------------------------------
# Heatmapa – pionowo czas, poziomo kolumny
# --------------------------------------------------------------------
plt.figure(figsize=(14, 8))

# Główna heatmapa
ax = sns.heatmap(
    missing_matrix,
    cmap="flare_r",           # czerwony = brak, jasny = obecne
    cbar_kws={'label': 'Brak danych (1 = brak)'},
    yticklabels=False,        # za dużo wierszy → etykiety y nieczytelne
    xticklabels=True,
    linewidths=0.01,
    linecolor='gray'
)

# Dodatkowe linie pionowe rozdzielające kolumny
ax.vlines(np.arange(0, missing_matrix.shape[1]+1), *ax.get_ylim(), color='gray', lw=0.8)

# Tytuł i podpisy
plt.title(f"{CFG.ticker} – Heatmapa brakujących wartości (OHLCV) w czasie", fontsize=14, pad=15)
plt.xlabel("Kolumna", fontsize=12)
plt.ylabel("Czas (każdy wiersz = jedna świeca dzienna)", fontsize=12)

# Mały trik – pokazujemy zakres dat na osi Y (tylko początek i koniec)
n_rows = len(df)
plt.yticks([0, n_rows-1], [df.index[0].strftime("%Y-%m-%d"), df.index[-1].strftime("%Y-%m-%d")])

# Statystyka w rogu
total_cells = missing_matrix.size
missing_cells = missing_matrix.sum().sum()
missing_pct = missing_cells / total_cells * 100

plt.text(
    0.02, 0.98,
    f"Brakuje {missing_cells:,} komórek\n({missing_pct:.3f}% wszystkich wartości)",
    transform=ax.transAxes,
    fontsize=11,
    va="top",
    ha="left",
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="gray")
)

plt.tight_layout()
plt.show()

# --------------------------------------------------------------------
# Dodatkowe szybkie podsumowanie brakujących po kolumnach
# --------------------------------------------------------------------
print("\nLiczba brakujących wartości w każdej kolumnie:")
missing_counts = missing_matrix.sum()
missing_percent = missing_matrix.mean() * 100
summary = pd.DataFrame({
    "Braki (szt.)": missing_counts,
    "% brakujących": missing_percent.round(4)
}).sort_values("Braki (szt.)", ascending=False)

print(summary)

if missing_counts.sum() == 0:
    print("\nBRAK BRAKÓW – dane są kompletne w całym zakresie.")
elif missing_counts["volume"] > 0 and missing_counts[["open","high","low","close"]].sum().sum() == 0:
    print("\nTylko volume ma braki – typowe dla niektórych źródeł danych.")

In [ ]:
# 4. Czyszczenie danych i ponowna walidacja

In [ ]:
# 5. Feature engineering

import numpy as np
import pandas as pd
from pathlib import Path

EPS = 1e-12


def add_log_returns(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    eps: float = EPS,
    prefix: str = "lr",
) -> pd.DataFrame:
    """
    Dodaje logarytmiczne stopy zwrotu (log-returns) względem poprzedniego zamknięcia.

    log_return(t) = log( (close_t + eps) / (close_{t-1} + eps) )

    Cechy są w pełni przyczynowe – używają wyłącznie przeszłych danych.

    Parameters
    ----------
    df : pd.DataFrame
        Dane wejściowe z kolumną cen zamknięcia
    close_col : str, default="close"
        Nazwa kolumny z ceną zamknięcia
    eps : float, default=1e-12
        Mała wartość zapobiegająca log(0) lub dzieleniu przez zero
    prefix : str, default="lr"
        Przedrostek dla nazwy nowej kolumny

    Returns
    -------
    pd.DataFrame
        Kopia df z dodaną kolumną {prefix}_1
    """
    out = df.copy()
    c = out[close_col].astype(float)
    out[f"{prefix}_1"] = np.log((c + eps) / (c.shift(1) + eps))
    return out


def add_candle_anatomy(
    df: pd.DataFrame,
    *,
    open_col: str = "open",
    high_col: str = "high",
    low_col: str = "low",
    close_col: str = "close",
    eps: float = EPS,
    prefix: str = "candle",
) -> pd.DataFrame:
    """
    Dodaje cechy opisujące anatomię świecy (body, wicki, kierunek) znormalizowane do zakresu świecy.

    Oblicza:
    - body_norm_rng        = |close - open| / (high - low + eps)
    - upper_wick_norm_rng  = (high - max(open,close)) / (high - low + eps)
    - lower_wick_norm_rng  = (min(open,close) - low) / (high - low + eps)
    - direction            = sign(close - open) ∈ {-1, 0, 1}

    Wszystkie cechy są obliczane wyłącznie na podstawie bieżącej świecy → zero leakage.

    Parameters
    ----------
    df : pd.DataFrame
    open_col, high_col, low_col, close_col : str
    eps : float
    prefix : str

    Returns
    -------
    pd.DataFrame
        Kopia df z nowymi kolumnami
    """
    out = df.copy()
    o = out[open_col].astype(float)
    h = out[high_col].astype(float)
    l = out[low_col].astype(float)
    c = out[close_col].astype(float)

    body = (c - o).abs()
    upper_wick = h - np.maximum(o, c)
    lower_wick = np.minimum(o, c) - l
    rng = (h - l).abs()
    denom = rng + eps

    out[f"{prefix}_body_norm_rng"] = body / denom
    out[f"{prefix}_upper_wick_norm_rng"] = upper_wick / denom
    out[f"{prefix}_lower_wick_norm_rng"] = lower_wick / denom
    out[f"{prefix}_direction"] = np.sign(c - o)

    return out


def _true_range(
    high: pd.Series,
    low: pd.Series,
    close: pd.Series,
) -> pd.Series:
    """Pomocnicza funkcja obliczająca True Range (Wilder)."""
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            (high - low).abs(),
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    return tr


def add_volatility_features(
    df: pd.DataFrame,
    *,
    high_col: str = "high",
    low_col: str = "low",
    close_col: str = "close",
    atr_window: int = 14,
    ret_std_window: int = 20,
    logret_col: str = "lr_1",
    prefix: str = "vol",
) -> pd.DataFrame:
    """
    Dodaje miary zmienności: ATR oraz odchylenie standardowe log-return.

    - ATR (Average True Range) – średnia krocząca True Range
    - ret_std – odchylenie standardowe log-return w oknie

    Obie miary są obliczane na danych przeszłych + bieżących → brak patrzenia w przyszłość.

    Parameters
    ----------
    df : pd.DataFrame
    high_col, low_col, close_col : str
    atr_window : int
    ret_std_window : int
    logret_col : str
        Nazwa kolumny z log-return (musi już istnieć)
    prefix : str

    Returns
    -------
    pd.DataFrame
    """
    out = df.copy()
    h = out[high_col].astype(float)
    l = out[low_col].astype(float)
    c = out[close_col].astype(float)

    tr = _true_range(h, l, c)
    atr = tr.rolling(atr_window, min_periods=atr_window).mean()
    out[f"{prefix}_atr_{atr_window}"] = atr

    if logret_col in out.columns:
        out[f"{prefix}_ret_std_{ret_std_window}"] = out[logret_col].rolling(
            ret_std_window, min_periods=ret_std_window
        ).std()
    else:
        out[f"{prefix}_ret_std_{ret_std_window}"] = np.nan

    return out


def add_volume_features(
    df: pd.DataFrame,
    *,
    volume_col: str = "volume",
    volume_window: int = 50,
    eps: float = EPS,
    prefix: str = "volm",
) -> pd.DataFrame:
    """
    Dodaje relatywny wolumen względem historycznej średniej (bez bieżącego wiersza).

    relative_volume(t) = volume(t) / mean(volume(t-volume_window : t-1) + eps)

    Średnia jest liczona wyłącznie z przeszłości → w pełni przyczynowa cecha.

    Parameters
    ----------
    df : pd.DataFrame
    volume_col : str
    volume_window : int
    eps : float
    prefix : str

    Returns
    -------
    pd.DataFrame
        Kopia df z kolumnami {prefix}_rel oraz {prefix}_log1p_rel
    """
    out = df.copy()
    v = out[volume_col].astype(float)

    # Średnia liczona TYLKO z przeszłości → shift(1) + rolling
    v_mean = v.shift(1).rolling(volume_window, min_periods=volume_window).mean()

    rel_v = v / (v_mean + eps)
    out[f"{prefix}_rel"] = rel_v
    out[f"{prefix}_log1p_rel"] = np.log1p(rel_v.clip(lower=0))

    return out


def add_features_v0_1(
    df: pd.DataFrame,
    *,
    open_col: str = "open",
    high_col: str = "high",
    low_col: str = "low",
    close_col: str = "close",
    volume_col: str = "volume",
    atr_window: int = 14,
    ret_std_window: int = 20,
    volume_window: int = 50,
    output_dir: str = "data/feature_engineering/",
    output_filename: str = "features_v0_1.csv",
) -> pd.DataFrame:
    """
    Główna funkcja agregująca – dodaje zestaw cech w wersji v0.1.

    Kolejność:
    1. log-returns
    2. anatomia świecy
    3. miary zmienności (ATR + std log-return)
    4. relatywny wolumen

    Zwraca kopię DataFrame z nowymi kolumnami i opcjonalnie zapisuje do CSV.

    Wersja v0.1 – podstawowy, przyczynowy zestaw cech do eksperymentów.

    Parameters
    ----------
    df : pd.DataFrame
    open_col, high_col, low_col, close_col, volume_col : str
    atr_window, ret_std_window, volume_window : int
    output_dir : str, default="data/feature_engineering/"
        Katalog do zapisu CSV
    output_filename : str, default="features_v0_1.csv"
        Nazwa pliku CSV

    Returns
    -------
    pd.DataFrame
        DataFrame z cechami
    """
    required = {open_col, high_col, low_col, close_col, volume_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Brakuje kolumn w df: {missing}")

    out = df.copy()

    # =========================
    # Krytyczne: porządek czasowy
    # =========================
    if "Date" in out.columns:
        out["Date"] = pd.to_datetime(out["Date"], errors="raise")
        if not out["Date"].is_monotonic_increasing:
            print("[WARNING] 'Date' nie jest rosnące. Sortuję rosnąco po Date.")
            out = out.sort_values("Date", ascending=True)

        # opcjonalnie: ustaw Date jako indeks (polecam w time-series)
        out = out.set_index("Date", drop=True)

    if isinstance(out.index, pd.DatetimeIndex):
        if not out.index.is_monotonic_increasing:
            print("[WARNING] Indeks czasu nie jest rosnący. Sortuję rosnąco po indeksie.")
            out = out.sort_index(ascending=True)

    if out.index.has_duplicates:
        raise ValueError("Indeks ma duplikaty dat – napraw to przed feature engineering.")

    out = add_log_returns(out, close_col=close_col, prefix="lr")
    out = add_candle_anatomy(
        out,
        open_col=open_col,
        high_col=high_col,
        low_col=low_col,
        close_col=close_col,
        prefix="candle",
    )
    out = add_volatility_features(
        out,
        high_col=high_col,
        low_col=low_col,
        close_col=close_col,
        atr_window=atr_window,
        ret_std_window=ret_std_window,
        logret_col="lr_1",
        prefix="vol",
    )
    out = add_volume_features(
        out,
        volume_col=volume_col,
        volume_window=volume_window,
        prefix="volm",
    )

    # Zapisz do CSV, jeśli podano ścieżkę
    output_path = Path(output_dir) / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_path, index=True)
    print(f"[INFO] Dane z cechami zapisane do: {output_path}")

    return out

In [ ]:
# =============================================================================
# Przykład użycia funkcji add_features_v0_1 z zapisem do CSV
# =============================================================================

# Zakładamy, że df już istnieje (wczytany w punkcie 1 lub po czyszczeniu w punkcie 4)
# print(df.head(3))          # możesz odkomentować, żeby zobaczyć surowe dane

print("[INFO] Przed dodaniem cech:")
print("Kształt:", df.shape)
print("Kolumny:", df.columns.tolist())

# -----------------------------------------------------------------------------
# Dodajemy cechy + zapisujemy wynik do CSV
# -----------------------------------------------------------------------------

df_features = add_features_v0_1(
    df=df,
    # parametry domyślne – można zmienić:
    # atr_window=14,
    # ret_std_window=20,
    # volume_window=50,
    
    # Najważniejsze – kontrola zapisu:
    output_dir="data/feature_engineering/",
    output_filename="GSPC_features_v0_1_2026-02-10.csv",   # ← sensowna nazwa z datą
)

# -----------------------------------------------------------------------------
# Krótki podgląd wyniku
# -----------------------------------------------------------------------------

print("\n[INFO] Po dodaniu cech:")
print("Kształt:", df_features.shape)
print("Nowe kolumny:", [col for col in df_features.columns if col not in df.columns])

print("\nPodgląd pierwszych 5 wierszy nowych cech:")
print(df_features.iloc[:5][[
    "lr_1",
    "candle_body_norm_rng",
    "candle_upper_wick_norm_rng",
    "candle_lower_wick_norm_rng",
    "candle_direction",
    "vol_atr_14",
    "vol_ret_std_20",
    "volm_rel",
    "volm_log1p_rel",
]])

# -----------------------------------------------------------------------------
# Sprawdzenie, czy plik naprawdę powstał
# -----------------------------------------------------------------------------

from pathlib import Path

saved_path = Path("data/feature_engineering/GSPC_features_v0_1_2026-02-10.csv")
if saved_path.exists():
    print(f"\n[OK] Plik zapisany pomyślnie: {saved_path}")
    print(f"Rozmiar pliku: {saved_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("\n[WARNING] Plik nie został zapisany – sprawdź uprawnienia do katalogu.")

In [ ]:
# =========================================
# 6. Walidacja cech z #5 (Feature engineering) - tutaj błąd w walidacji - powinno wyłapać NaN
# - ARCHITEKTURA_VALIDATION_RUNNER_V1
# =========================================

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence
import numpy as np
import pandas as pd


# =========================
# Wyjątki / wyniki walidacji
# =========================

class ValidationError(ValueError):
    """Błąd walidacji danych / cech (fail-fast)."""


@dataclass(frozen=True)
class ValidationIssue:
    validator: str
    check: str
    message: str
    n_bad: int = 0
    csv_path: Optional[Path] = None


# =========================
# Baza dla walidatorów cech
# =========================

class BaseFeatureValidator:
    """Interfejs dla pojedynczego walidatora cech (po FE)."""
    name: str = "BaseFeatureValidator"

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        raise NotImplementedError

    def _ensure_blacksheep_dir(self, blacksheep_dir: Optional[Path]) -> Path:
        if blacksheep_dir is None:
            raise ValidationError("blacksheep_dir jest wymagany gdy saveBlackSheeps=True.")
        d = Path(blacksheep_dir)
        d.mkdir(parents=True, exist_ok=True)
        return d

    def _save_blacksheeps(
        self,
        df_or_subset: pd.DataFrame,
        *,
        blacksheep_dir: Path,
        check: str,
    ) -> Path:
        safe_check = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in check)
        out_path = blacksheep_dir / f"{self.name}__{safe_check}.csv"
        df_or_subset.to_csv(out_path, index=True)
        return out_path


# =========================
# Runner (orkiestrator)
# =========================

class FeatureValidationRunner:
    """
    Orkiestrator walidatorów cech:
    - kompozycja walidatorów
    - fail-fast (ValidationError zatrzymuje pipeline)
    - runner nie modyfikuje df
    """
    def __init__(self, validators: Sequence[BaseFeatureValidator]):
        self.validators = list(validators)

    def run(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
        test_run: bool = False,   # <-- NOWE
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        # =========================
        # TEST MODE: wstrzyknięcie leaky add_log_returns
        # =========================
        if test_run:
            if "add_log_returns" not in globals():
                raise ValidationError("test_run=True, ale add_log_returns nie istnieje w globals().")

            _add_log_returns_original = globals()["add_log_returns"]

            def add_log_returns_LEAKY(
                df_in: pd.DataFrame,
                *,
                close_col: str = "close",
                prefix: str = "lr",
            ) -> pd.DataFrame:
                out = df_in.copy()
                # CELOWY WYCIEK: używamy przyszłej ceny
                out[f"{prefix}_1"] = np.log(
                    (out[close_col].shift(-1) + 1e-12) / (out[close_col] + 1e-12)
                )
                return out

            globals()["add_log_returns"] = add_log_returns_LEAKY

            try:
                for v in self.validators:
                    issues.extend(
                        v.validate(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir)
                    )
            finally:
                # zawsze przywróć oryginał, nawet jeśli walidator rzuci wyjątek
                globals()["add_log_returns"] = _add_log_returns_original

            return issues

        # =========================
        # NORMAL MODE
        # =========================
        for v in self.validators:
            issues.extend(v.validate(df, saveBlackSheeps=saveBlackSheeps, blacksheep_dir=blacksheep_dir))
        return issues


# ==========================================================
# Walidator: Data Leakage dla cech z #5 (v0.1)
# ==========================================================

class FeatureDataLeakageValidatorV0_1(BaseFeatureValidator):
    """
    Detekcja leakage dla cech v0.1 przez REKOMPUTACJĘ cech (deterministycznie)
    i porównanie z tym co jest w df.

    Jeśli cechy w df nie odpowiadają przyczynowej implementacji z punktu #5
    → traktujemy to jako potencjalny leakage / błąd pipeline → ValidationError.

    Wymaga aby w środowisku były zdefiniowane funkcje z punktu #5:
    - add_log_returns
    - add_candle_anatomy
    - add_volatility_features
    - add_volume_features
    """
    name = "FeatureDataLeakageValidatorV0_1"

    def __init__(
        self,
        *,
        open_col: str = "open",
        high_col: str = "high",
        low_col: str = "low",
        close_col: str = "close",
        volume_col: str = "volume",
        atr_window: int = 14,
        ret_std_window: int = 20,
        volume_window: int = 50,
        atol: float = 1e-10,
        rtol: float = 1e-8,
    ):
        self.open_col = open_col
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col
        self.volume_col = volume_col

        self.atr_window = atr_window
        self.ret_std_window = ret_std_window
        self.volume_window = volume_window

        self.atol = float(atol)
        self.rtol = float(rtol)

        self.expected_feature_cols = [
            "lr_1",
            "candle_body_norm_rng",
            "candle_upper_wick_norm_rng",
            "candle_lower_wick_norm_rng",
            "candle_direction",
            f"vol_atr_{atr_window}",
            f"vol_ret_std_{ret_std_window}",
            "volm_rel",
            "volm_log1p_rel",
        ]

    def _recompute_features(self, df: pd.DataFrame) -> pd.DataFrame:
        # Funkcje MUSZĄ istnieć (z punktu #5)
        required_fns = [
            "add_log_returns",
            "add_candle_anatomy",
            "add_volatility_features",
            "add_volume_features",
        ]
        missing_fns = [fn for fn in required_fns if fn not in globals()]
        if missing_fns:
            raise ValidationError(
                "Brakuje funkcji z #5 w aktualnym scope: "
                f"{missing_fns}. Uruchom komórkę z Feature engineering przed walidacją."
            )

        raw_required = {self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col}
        missing_cols = raw_required - set(df.columns)
        if missing_cols:
            raise ValidationError(f"Brakuje kolumn surowych do rekomputacji cech: {missing_cols}")

        out = df[[self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col]].copy()

        # 1) log-returns
        out = add_log_returns(out, close_col=self.close_col, prefix="lr")

        # 2) anatomia świecy
        out = add_candle_anatomy(
            out,
            open_col=self.open_col,
            high_col=self.high_col,
            low_col=self.low_col,
            close_col=self.close_col,
            prefix="candle",
        )

        # 3) volatility
        out = add_volatility_features(
            out,
            high_col=self.high_col,
            low_col=self.low_col,
            close_col=self.close_col,
            atr_window=self.atr_window,
            ret_std_window=self.ret_std_window,
            logret_col="lr_1",
            prefix="vol",
        )

        # 4) volume (past-only)
        out = add_volume_features(
            out,
            volume_col=self.volume_col,
            volume_window=self.volume_window,
            prefix="volm",
        )

        return out

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        missing_feat = [c for c in self.expected_feature_cols if c not in df.columns]
        if missing_feat:
            raise ValidationError(
                f"Brakuje oczekiwanych kolumn cech v0.1 w df: {missing_feat}. "
                "Najpierw policz cechy z #5 (add_features_v0_1 lub równoważnie)."
            )

        df_re = self._recompute_features(df)

        # Porównanie kolumn: tolerancja numeryczna, NaN==NaN
        bad_masks = {}
        for col in self.expected_feature_cols:
            a = df[col].astype(float)
            b = df_re[col].astype(float)
            same = np.isclose(a.values, b.values, rtol=self.rtol, atol=self.atol, equal_nan=True)
            bad = ~same
            if bad.any():
                bad_masks[col] = bad

        if bad_masks:
            # jeden zbiorczy issue (fail-fast), ale zliczamy ile rekordów jest "podejrzanych"
            # union po wszystkich cechach
            union_bad = np.zeros(len(df), dtype=bool)
            for m in bad_masks.values():
                union_bad |= m

            n_bad = int(union_bad.sum())
            msg = (
                "Wykryto rozbieżność między cechami w df a przyczynową rekomputacją v0.1. "
                "To zwykle oznacza błąd pipeline albo potencjalny data leakage "
                "(np. cechy policzone z użyciem przyszłych wartości lub innej implementacji). "
                f"N_podejrzanych_wierszy={n_bad}, kolumny={list(bad_masks.keys())}"
            )

            csv_path = None
            if saveBlackSheeps:
                d = self._ensure_blacksheep_dir(blacksheep_dir)
                cols_preview = [
                    self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col,
                    *self.expected_feature_cols,
                ]
                preview = df.loc[union_bad, cols_preview].copy()
                # dorzucamy recomputed obok (suffix), żeby było widać różnice
                for col in self.expected_feature_cols:
                    preview[f"{col}__recomputed"] = df_re.loc[union_bad, col].values
                csv_path = self._save_blacksheeps(preview, blacksheep_dir=d, check="feature_leakage_recompute_mismatch")

            issues.append(
                ValidationIssue(
                    validator=self.name,
                    check="recompute_compare_v0_1",
                    message=msg,
                    n_bad=n_bad,
                    csv_path=csv_path,
                )
            )

            # FAIL-FAST
            raise ValidationError(msg)

        # jeśli chcesz mimo wszystko zwracać info "OK", zostawiamy pusto (jak w Raw runnerze)
        return issues

from __future__ import annotations

from pathlib import Path
from typing import Optional, Sequence
import numpy as np
import pandas as pd


class FeatureIncrementalConsistencyValidatorV0_1(BaseFeatureValidator):
    """
    Incremental Consistency Test (bardzo mocny test przyczynowości):
    - licz cechy na prefixie danych (N)
    - licz cechy na większym prefixie (N+K)
    - porównaj wartości cech dla pierwszych N wierszy
      => muszą być identyczne, jeśli FE jest przyczynowe i deterministyczne
    """
    name = "FeatureIncrementalConsistencyValidatorV0_1"

    def __init__(
        self,
        *,
        open_col: str = "open",
        high_col: str = "high",
        low_col: str = "low",
        close_col: str = "close",
        volume_col: str = "volume",
        atr_window: int = 14,
        ret_std_window: int = 20,
        volume_window: int = 50,
        cutoffs: Sequence[int] = (60, 120, 250),
        step: int = 60,
        atol: float = 1e-10,
        rtol: float = 1e-8,
        max_rows_in_blacksheep: int = 200,
    ):
        self.open_col = open_col
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col
        self.volume_col = volume_col

        self.atr_window = atr_window
        self.ret_std_window = ret_std_window
        self.volume_window = volume_window

        self.cutoffs = list(int(x) for x in cutoffs)
        self.step = int(step)

        self.atol = float(atol)
        self.rtol = float(rtol)

        self.max_rows_in_blacksheep = int(max_rows_in_blacksheep)

        self.feature_cols = [
            "lr_1",
            "candle_body_norm_rng",
            "candle_upper_wick_norm_rng",
            "candle_lower_wick_norm_rng",
            "candle_direction",
            f"vol_atr_{atr_window}",
            f"vol_ret_std_{ret_std_window}",
            "volm_rel",
            "volm_log1p_rel",
        ]

    def _compute_features_prefix(self, df: pd.DataFrame) -> pd.DataFrame:
        # Wymagamy funkcji z punktu #5
        required_fns = [
            "add_log_returns",
            "add_candle_anatomy",
            "add_volatility_features",
            "add_volume_features",
        ]
        missing_fns = [fn for fn in required_fns if fn not in globals()]
        if missing_fns:
            raise ValidationError(
                "Brakuje funkcji z #5 w aktualnym scope: "
                f"{missing_fns}. Uruchom komórkę z Feature engineering przed walidacją."
            )

        raw_required = {self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col}
        missing_cols = raw_required - set(df.columns)
        if missing_cols:
            raise ValidationError(f"Brakuje kolumn surowych do FE: {missing_cols}")

        out = df[[self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col]].copy()

        out = add_log_returns(out, close_col=self.close_col, prefix="lr")

        out = add_candle_anatomy(
            out,
            open_col=self.open_col,
            high_col=self.high_col,
            low_col=self.low_col,
            close_col=self.close_col,
            prefix="candle",
        )

        out = add_volatility_features(
            out,
            high_col=self.high_col,
            low_col=self.low_col,
            close_col=self.close_col,
            atr_window=self.atr_window,
            ret_std_window=self.ret_std_window,
            logret_col="lr_1",
            prefix="vol",
        )

        out = add_volume_features(
            out,
            volume_col=self.volume_col,
            volume_window=self.volume_window,
            prefix="volm",
        )

        return out[self.feature_cols].copy()

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        n_total = len(df)
        if n_total < (min(self.cutoffs) + self.step):
            raise ValidationError(
                f"Za mało danych do incremental consistency: len(df)={n_total}, "
                f"min_cutoff+step={min(self.cutoffs)+self.step}."
            )

        # sprawdź, czy df ma już te feature_cols (opcjonalnie)
        missing_feat = [c for c in self.feature_cols if c not in df.columns]
        if missing_feat:
            # To nie jest krytyczne dla samego testu incremental (bo recomputujemy),
            # ale zwykle oznacza, że uruchamiasz walidację w złym miejscu pipeline.
            raise ValidationError(
                f"Brakuje kolumn cech v0.1 w df: {missing_feat}. "
                "Uruchom walidację po #5 Feature engineering."
            )

        for cutoff in self.cutoffs:
            n1 = int(cutoff)
            n2 = int(cutoff + self.step)
            if n2 > n_total:
                continue

            df1 = df.iloc[:n1]
            df2 = df.iloc[:n2]

            f1 = self._compute_features_prefix(df1)
            f2 = self._compute_features_prefix(df2).iloc[:n1]

            # Porównaj pierwsze N wierszy: MUSZĄ być identyczne
            bad_any = np.zeros(n1, dtype=bool)
            bad_cols: list[str] = []

            for col in self.feature_cols:
                a = f1[col].astype(float).values
                b = f2[col].astype(float).values
                same = np.isclose(a, b, rtol=self.rtol, atol=self.atol, equal_nan=True)
                bad = ~same
                if bad.any():
                    bad_any |= bad
                    bad_cols.append(col)

            if bad_any.any():
                n_bad = int(bad_any.sum())
                msg = (
                    "Incremental consistency FAILED: cechy zmieniają się wstecz po dopisaniu przyszłych danych. "
                    f"cutoff={n1}, cutoff_plus={n2}, n_bad={n_bad}, cols={bad_cols}. "
                    "To jest silny sygnał nieprzyczynowej transformacji / data leakage "
                    "(np. shift(-1), rolling center=True, globalne statystyki, fit na całym zbiorze)."
                )

                csv_path = None
                if saveBlackSheeps:
                    d = self._ensure_blacksheep_dir(blacksheep_dir)

                    idx_bad = np.where(bad_any)[0]
                    idx_bad = idx_bad[: self.max_rows_in_blacksheep]

                    preview = df.iloc[idx_bad][
                        [self.open_col, self.high_col, self.low_col, self.close_col, self.volume_col] + self.feature_cols
                    ].copy()

                    # dorzuć wartości FE z prefixu N oraz z prefixu N+K (dla porównania)
                    for col in self.feature_cols:
                        preview[f"{col}__prefix_{n1}"] = f1.iloc[idx_bad][col].values
                        preview[f"{col}__prefix_{n2}"] = f2.iloc[idx_bad][col].values

                    csv_path = self._save_blacksheeps(
                        preview,
                        blacksheep_dir=d,
                        check=f"incremental_consistency__{n1}_vs_{n2}",
                    )

                issues.append(
                    ValidationIssue(
                        validator=self.name,
                        check=f"prefix_{n1}_vs_{n2}",
                        message=msg,
                        n_bad=n_bad,
                        csv_path=csv_path,
                    )
                )

                # FAIL-FAST
                raise ValidationError(msg)

        return issues

# ==========================================================
# Walidator: Statistical Leakage (feature -> future target)
# ==========================================================

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence
import numpy as np
import pandas as pd


class FeatureStatisticalLeakageValidatorV0_1(BaseFeatureValidator):
    """
    Statistical leakage test:
    - buduje target jako future log-return (horizon h) z close.shift(-h)
    - sprawdza korelacje feature->target (i opcjonalnie mutual information)
    - fail-fast jeśli wykryje podejrzanie wysoką zależność

    To NIE jest test deterministyczności FE.
    To jest test "czy cecha wygląda jak target / przyszłość".
    """
    name = "FeatureStatisticalLeakageValidatorV0_1"

    def __init__(
        self,
        *,
        feature_cols: Sequence[str],
        close_col: str = "close",
        horizon: int = 1,
        eps: float = 1e-12,
        corr_threshold: float = 0.95,
        top_k_report: int = 30,
        compute_mutual_info: bool = False,
        mi_threshold: float = 0.5,
        n_bins_for_mi: int = 30,
        min_non_nan: int = 10,
    ):
        self.feature_cols = list(feature_cols)
        self.close_col = close_col
        self.horizon = int(horizon)
        if self.horizon <= 0:
            raise ValueError("horizon musi być >= 1.")
        self.eps = float(eps)

        self.corr_threshold = float(corr_threshold)
        self.top_k_report = int(top_k_report)

        self.compute_mutual_info = bool(compute_mutual_info)
        self.mi_threshold = float(mi_threshold)
        self.n_bins_for_mi = int(n_bins_for_mi)

        self.min_non_nan = int(min_non_nan)

    def _make_target(self, df: pd.DataFrame) -> pd.Series:
        if self.close_col not in df.columns:
            raise ValidationError(f"Brakuje close_col='{self.close_col}' do budowy targetu.")
        c = df[self.close_col].astype(float)
        # future log return: log(close[t+h]/close[t])
        y = np.log((c.shift(-self.horizon) + self.eps) / (c + self.eps))
        y.name = f"future_log_return_{self.horizon}"
        return y

    def _mutual_info_discrete(self, x: np.ndarray, y: np.ndarray, n_bins: int) -> float:
        """
        Prosty MI przez dyskretyzację do binów (deterministyczny, bez sklearn).
        MI(X;Y) = sum p(x,y) log( p(x,y) / (p(x)p(y)) )
        """
        # binning przez kwantyle (bardziej stabilne dla ogonów)
        def qcut_bins(a: np.ndarray, bins: int) -> np.ndarray:
            qs = np.linspace(0, 1, bins + 1)
            edges = np.quantile(a, qs)
            # zabezpieczenie przed powtarzającymi się krawędziami
            edges = np.unique(edges)
            if len(edges) <= 2:
                # praktycznie stała zmienna
                return np.zeros_like(a, dtype=int)
            # digitize: 1..len(edges)-1
            b = np.digitize(a, edges[1:-1], right=True)
            return b.astype(int)

        xb = qcut_bins(x, n_bins)
        yb = qcut_bins(y, n_bins)

        # joint histogram
        xk = xb.max() + 1
        yk = yb.max() + 1
        joint = np.zeros((xk, yk), dtype=float)
        for xi, yi in zip(xb, yb):
            joint[xi, yi] += 1.0
        joint /= joint.sum()

        px = joint.sum(axis=1, keepdims=True)
        py = joint.sum(axis=0, keepdims=True)

        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = joint / (px @ py)
            mi = np.nansum(joint * np.log(ratio))
        if not np.isfinite(mi):
            mi = 0.0
        return float(mi)

    def validate(
        self,
        df: pd.DataFrame,
        *,
        saveBlackSheeps: bool = False,
        blacksheep_dir: Optional[Path] = None,
    ) -> list[ValidationIssue]:
        issues: list[ValidationIssue] = []

        missing = [c for c in self.feature_cols if c not in df.columns]
        if missing:
            raise ValidationError(f"Brakuje feature_cols w df: {missing}")

        y = self._make_target(df)

        rows = []
        for col in self.feature_cols:
            x = df[col].astype(float)
            mask = x.notna() & y.notna()
            n = int(mask.sum())
            if n < self.min_non_nan:
                continue

            xv = x[mask].values
            yv = y[mask].values

            corr = float(np.corrcoef(xv, yv)[0, 1]) if n >= 2 else np.nan
            abs_corr = float(abs(corr)) if np.isfinite(corr) else np.nan

            mi = np.nan
            if self.compute_mutual_info:
                mi = self._mutual_info_discrete(xv, yv, self.n_bins_for_mi)

            rows.append(
                {
                    "feature": col,
                    "n": n,
                    "corr": corr,
                    "abs_corr": abs_corr,
                    "mutual_info": mi,
                }
            )

        if not rows:
            raise ValidationError(
                "Statistical leakage validator: brak wystarczającej liczby obserwacji "
                f"(min_non_nan={self.min_non_nan})."
            )

        report = pd.DataFrame(rows).sort_values("abs_corr", ascending=False)

        # reguły alarmowe
        bad_corr = report["abs_corr"].notna() & (report["abs_corr"] >= self.corr_threshold)
        bad_mi = False
        if self.compute_mutual_info:
            bad_mi = report["mutual_info"].notna() & (report["mutual_info"] >= self.mi_threshold)

        flagged = report[bad_corr | bad_mi].copy()

        if not flagged.empty:
            msg = (
                "Wykryto podejrzanie silny związek feature -> future target "
                f"({y.name}). To może oznaczać data leakage lub cechę bliską definicji targetu. "
                f"Najgorsze: {flagged.head(min(5, len(flagged)))['feature'].tolist()}."
            )

            csv_path = None
            if saveBlackSheeps:
                d = self._ensure_blacksheep_dir(blacksheep_dir)
                out = report.head(self.top_k_report).copy()
                out.insert(0, "target", y.name)
                out_path = d / f"{self.name}__ranking_{y.name}.csv"
                out.to_csv(out_path, index=False)
                csv_path = out_path

            issues.append(
                ValidationIssue(
                    validator=self.name,
                    check="feature_to_future_target_dependency",
                    message=msg,
                    n_bad=int(len(flagged)),
                    csv_path=csv_path,
                )
            )

            raise ValidationError(msg)

        return issues


# =========================
# Przykład użycia
# =========================

# Do testów czy się wywali---------------------
# StatisticalLeakageValidator:
#df_bad = df_features.copy()

#h = 1
#df_bad["leak_future_lr"] = np.log((df_bad["close"].shift(-h) + 1e-12) / (df_bad["close"] + 1e-12))

# FeatureDataLeakageValidatorV0_1: - to zadziałało
#df_bad = df_features.copy()
#df_bad["lr_1"] = df_bad["lr_1"] + 0.001  # celowe popsucie

# FeatureIncrementalConsistencyValidatorV0_1:
#df_bad = df_features.copy()
# Cecha, która ZMIENI wartości historyczne po dopisaniu przyszłości:
#df_bad["leak_centered_mean"] = df_bad["close"].rolling(11, center=True).mean()
#---------------------

feature_cols_v0_1 = [
    "lr_1",
    "candle_body_norm_rng",
    "candle_upper_wick_norm_rng",
    "candle_lower_wick_norm_rng",
    "candle_direction",
    "vol_atr_14",
    "vol_ret_std_20",
    "volm_rel",
    "volm_log1p_rel",
]

# Do testów czy się wywali---------------------
# StatisticalLeakageValidator:
#feature_cols_for_leak_test = [*feature_cols_v0_1, "leak_future_lr"]
#---------------------

FEATURE_BLACKSHEEP_DIR = Path("data/validation/feature_blacksheeps")

feature_runner = FeatureValidationRunner(
    validators=[
        FeatureDataLeakageValidatorV0_1(atr_window=14, ret_std_window=20, volume_window=50),
        FeatureIncrementalConsistencyValidatorV0_1(
            atr_window=14,
            ret_std_window=20,
            volume_window=50,
            cutoffs=(10, 20, 60, 120),  # dokładnie jak pytałeś: 10 vs 10+step, itd.
            step=10,                   # np. 10 dni
        ),
        FeatureStatisticalLeakageValidatorV0_1(
            # Do testów czy się wywali---------------------
            # StatisticalLeakageValidator:
            # feature_cols=feature_cols_for_leak_test,
            #---------------------
            feature_cols=feature_cols_v0_1,
            close_col="close",
            horizon=1,                 # target: future_log_return_1
            corr_threshold=0.95,       # trading: 0.95 to już prawie pewny wyciek
            compute_mutual_info=False, # możesz włączyć później
        )
    ]
)

try:
    # Do testów czy się wywali---------------------
    #feature_runner.run(df_bad, saveBlackSheeps=True, blacksheep_dir=FEATURE_BLACKSHEEP_DIR)
    #---------------------
    feature_runner.run(df_features, saveBlackSheeps=True, blacksheep_dir=FEATURE_BLACKSHEEP_DIR)
    #feature_runner.run(df_bad, saveBlackSheeps=True, blacksheep_dir=FEATURE_BLACKSHEEP_DIR, test_run=True)
    print("[OK] Feature validation passed.")
except ValidationError:
    print("[ERROR] Feature validation failed.")
    raise


In [ ]:
# =========================
# 7. Podział danych (train / val / test) + zapis CSV
# =========================

from dataclasses import dataclass
from pathlib import Path
import pandas as pd


class DataSplitError(ValueError):
    """Błąd podziału danych."""


@dataclass(frozen=True)
class TimeSeriesSplitConfig:
    train_ratio: float = 0.7
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    def validate(self):
        total = self.train_ratio + self.val_ratio + self.test_ratio
        if abs(total - 1.0) > 1e-6:
            raise DataSplitError(
                f"Suma proporcji musi wynosić 1.0, obecnie: {total}"
            )


def time_series_train_val_test_split(
    df: pd.DataFrame,
    *,
    config: TimeSeriesSplitConfig = TimeSeriesSplitConfig(),
    save_dir: str | Path | None = None,
    file_prefix: str = "dataset"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    if not isinstance(df.index, pd.DatetimeIndex):
        raise DataSplitError("Index musi być typu DatetimeIndex.")

    if df.isnull().any().any():
        raise DataSplitError("Dane zawierają NaN – najpierw wykonaj czyszczenie.")

    config.validate()

    n = len(df)
    if n < 100:
        raise DataSplitError("Zbyt mało obserwacji do bezpiecznego podziału.")

    train_end = int(n * config.train_ratio)
    val_end = train_end + int(n * config.val_ratio)

    train = df.iloc[:train_end].copy()
    val = df.iloc[train_end:val_end].copy()
    test = df.iloc[val_end:].copy()

    if len(test) == 0:
        raise DataSplitError("Test set jest pusty.")

    print("=== PODZIAŁ DANYCH ===")
    print(f"Train: {len(train)} ({len(train)/n:.2%})")
    print(f"Val  : {len(val)} ({len(val)/n:.2%})")
    print(f"Test : {len(test)} ({len(test)/n:.2%})")

    # =========================
    # Zapis do CSV
    # =========================
    if save_dir is not None:
        save_path = Path(save_dir)
        save_path.mkdir(parents=True, exist_ok=True)

        train_path = save_path / f"{file_prefix}_train.csv"
        val_path = save_path / f"{file_prefix}_val.csv"
        test_path = save_path / f"{file_prefix}_test.csv"

        train.to_csv(train_path)
        val.to_csv(val_path)
        test.to_csv(test_path)

        print("\n=== ZAPISANO PLIKI CSV ===")
        print(train_path)
        print(val_path)
        print(test_path)

    return train, val, test


In [ ]:
train_df, val_df, test_df = time_series_train_val_test_split(
    df_features,
    save_dir="data/splits",
    file_prefix="market_features"
)

In [ ]:
# Usunięcie danych gdzie feature cechy są puste - jeśli to na górze się wywali

df_cleaned = drop_rows_by_cutoff_date_to_csv(
    df_features,
    date_col="Date",
    cutoff_date="1962-03-14",
    remove="below",
    out_dir="data/validation",
)


In [ ]:
train_df, val_df, test_df = time_series_train_val_test_split(
    df_cleaned,
    save_dir="data/splits",
    file_prefix="market_features"
)

In [ ]:
# ============================================
# 9. Budowa modelu (prosta wersja, 100% kroku 9) - tutaj bym zostawił
# - tworzymy target (future log return)
# - opcjonalnie zamieniamy go na klasyfikację (up/down)
# - wybieramy kolumny cech
# - usuwamy NaN w target (bo shift(-horizon) robi NaN na końcu)
# - budujemy preprocessor (num: median+scaler, cat: most_frequent+onehot)
# - wybieramy model (log_reg / ridge / rf / hgb)
# - składamy Pipeline: preprocessor -> model
# - zapisujemy spec (konfiguracja + feature_cols + target_name)
# ============================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor


# ============================================
# 9.0 Ustawienia (proste, jawne)
# ============================================

TASK = "classification"        # "classification" albo "regression"
MODEL_KIND = "hgb"             # dla classification: "log_reg" / "rf" / "hgb"
                               # dla regression:     "ridge" / "rf" / "hgb"

RANDOM_STATE = 42
N_JOBS = -1

ARTIFACTS_DIR = Path("data/modeling/build_model")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CLOSE_COL = "close"
HORIZON = 1                    # ile dni do przodu liczymy target
THRESHOLD = 0.0                # tylko dla classification: y=1 jeśli log_return > threshold


# ============================================
# 9.1 Walidacja wejścia (minimum, żeby nie wysadzić się w runtime)
# ============================================

def _check_df(df: pd.DataFrame, name: str):
    if df is None or len(df) == 0:
        raise ValueError(f"{name} jest puste lub None.")

def _check_column(df: pd.DataFrame, col: str, df_name: str):
    if col not in df.columns:
        raise ValueError(f"W {df_name} brakuje kolumny '{col}'.")


# ============================================
# 9.2 Target (future log return) + opcjonalna klasyfikacja
# ============================================
# To są targety czyli definiujemy tutaj co model ma się uczyć, targety są dwa w kodzie ponieważ chcemy żeby model najpierw nauczył się dla jednego targetu (czyli jeden cały proces), następnie drugiego i wykonamy porównanie w backteście (symulacja realnego handlu na danych testowych (krok po kroku w czasie))

def make_future_log_return(df: pd.DataFrame, close_col: str, horizon: int, out_name: str):
    """
    y[t] = log( Close[t+h] / Close[t] )
    """
    close = pd.to_numeric(df[close_col], errors="coerce")
    if close.isna().all():
        raise ValueError(f"Kolumna '{close_col}' po konwersji jest cała NaN.")
    y = np.log(close.shift(-horizon) / close)
    y.name = out_name
    return y

def make_binary_target(y_reg: pd.Series, threshold: float, out_name: str):
    """
    y=1 jeśli y_reg > threshold, inaczej 0
    """
    if y_reg is None or len(y_reg) == 0:
        raise ValueError("y_reg jest puste lub None.")
    y = (y_reg > threshold).astype("int64")
    y.name = out_name
    return y


# ============================================
# 9.3 Wybór cech: prosto (wykluczamy OHLCV, Date i targety)
# ============================================

def infer_feature_columns(df: pd.DataFrame, exclude_cols):
    exclude_set = set(exclude_cols)
    feature_cols = [c for c in df.columns if c not in exclude_set]
    if len(feature_cols) == 0:
        raise ValueError("Nie udało się wyznaczyć żadnych kolumn cech.")
    return feature_cols


# ============================================
# 9.4 Usunięcie NaN w target (ostatnie wiersze przez shift(-horizon))
# ============================================

def drop_nan_target_rows(X: pd.DataFrame, y: pd.Series):
    if len(X) != len(y):
        raise ValueError(f"Niezgodne długości: len(X)={len(X)} vs len(y)={len(y)}")
    mask = ~y.isna()
    X2 = X.loc[mask].copy()
    y2 = y.loc[mask].copy()
    if len(X2) == 0:
        raise ValueError("Po usunięciu NaN w target zostało 0 wierszy.")
    return X2, y2


# ============================================
# 9.5 Preprocessor (Géron-style)
# - num: median -> standard scaler
# - cat: most_frequent -> onehot(ignore unknown)
# ============================================
# Buduje mechanizm, który automatycznie czyści i przekształca X (podzielone dane z ptk 7) tak, aby model mógł na nim wykonywać obliczenia.
# Co dokładnie robi krok po kroku?
# * Rozdziela kolumny
# ** wykrywa kolumny numeryczne
# ** wykrywa kolumny kategoryczne
# 
# Dla kolumn numerycznych:
# * Tworzy pipeline:
# ** uzupełnia NaN medianą
# ** skaluje dane (StandardScaler)
# 
# Efekt:
# * brak NaN
# * wartości w stabilnej skali
# 
# Dla kolumn kategorycznych:
# * Tworzy pipeline:
# ** uzupełnia NaN najczęstszą wartością
# ** zamienia tekst na liczby (OneHotEncoder)
# 
# Efekt:
# * brak tekstów
# * wszystko w postaci liczb
# 
# Łączy wszystko w ColumnTransformer
# Czyli:
# * do num → zastosuj num_pipeline
# * do cat → zastosuj cat_pipeline
# * połącz w jedną macierz liczb

def build_preprocessor(X: pd.DataFrame):
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    if len(numeric_cols) == 0 and len(categorical_cols) == 0:
        raise ValueError("Brak kolumn do preprocessingu.")

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", num_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", cat_pipe, categorical_cols))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return preprocessor


# ============================================
# 9.6 Wybór modelu (jawny if, bez abstrakcji)
# ============================================
# Wybiera i tworzy konkretny algorytm ML odpowiedni do zadania (regresja lub klasyfikacja), ale go jeszcze nie trenuje.

def build_model(task: str, model_kind: str):
    if task == "classification":
        if model_kind == "log_reg":
            return LogisticRegression(max_iter=2000, n_jobs=N_JOBS, random_state=RANDOM_STATE) # dowiedzieć się o tych parametrach co to jest
        if model_kind == "rf":
            return RandomForestClassifier(
                n_estimators=400, max_depth=None, n_jobs=N_JOBS, random_state=RANDOM_STATE
            )
        if model_kind == "hgb":
            return HistGradientBoostingClassifier(
                max_depth=None, learning_rate=0.05, max_iter=500, random_state=RANDOM_STATE # dowiedzieć się o tych parametrach co to jest i czemu musi być tutaj
            )
        raise ValueError(f"Nieznany MODEL_KIND dla classification: {model_kind}")

    if task == "regression":
        if model_kind == "ridge":
            return Ridge(alpha=1.0, random_state=RANDOM_STATE) # dowiedzieć się o tych parametrach co to jest i czemu musi być tutaj
        if model_kind == "rf":
            return RandomForestRegressor(
                n_estimators=400, max_depth=None, n_jobs=N_JOBS, random_state=RANDOM_STATE
            )
        if model_kind == "hgb":
            return HistGradientBoostingRegressor(
                max_depth=None, learning_rate=0.05, max_iter=800, random_state=RANDOM_STATE
            )
        raise ValueError(f"Nieznany MODEL_KIND dla regression: {model_kind}")

    raise ValueError(f"Nieznany TASK: {task}")


# ============================================
# 9.7 Składamy pipeline: preprocessor -> model (bez fit!)
# ============================================
# Tworzy gotowy obiekt, który łączy preprocessing i model w jeden spójny mechanizm uczenia.
# (preprocessor) → (model)
# 

def build_pipeline(X_sample: pd.DataFrame, task: str, model_kind: str, use_preprocessor: bool = True):
    if use_preprocessor:
        preprocessor = build_preprocessor(X_sample)
        model = build_model(task, model_kind)
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])
        return pipe
    else:
        # opcja gdy masz już gotowe X_scaled z punktu 8 i nie chcesz preprocessora w pipeline
        model = build_model(task, model_kind)
        pipe = Pipeline([
            ("model", model),
        ])
        return pipe


# ============================================
# 9.8 Zapis artefaktów kroku 9 (specyfikacja budowy)
# - zapisujemy: ustawienia + feature_cols + target_name
# (bez wag modelu, bo nie ma fit)
# ============================================

def save_model_build_spec(path: Path, spec: dict):
    path.write_text(json.dumps(spec, indent=2, ensure_ascii=False))
    print(f"[INFO] Zapisano: {path}")


# ============================================
# 9.8a ROZSZERZONY KOD - Odtworzenie pipeline z zapisanej specyfikacji (model_build_spec.json)
# ============================================

def load_model_build_spec(spec_path: Path) -> dict:
    """
    Wczytuje specyfikację budowy modelu zapisaną w punkcie 9.
    Uwaga: to NIE jest wytrenowany model, tylko opis jak go zbudować.
    """
    if not spec_path.exists():
        raise FileNotFoundError(f"Nie znaleziono pliku specyfikacji: {spec_path}")
    return json.loads(spec_path.read_text())


def build_pipeline_from_spec(X_sample: pd.DataFrame, spec: dict) -> Pipeline:
    """
    Odtwarza pipeline (preprocessor -> model) z dict spec.
    Pipeline będzie w stanie 'unfitted' (brak fit).
    """
    if X_sample is None or len(X_sample) == 0:
        raise ValueError("X_sample jest puste lub None.")
    if not isinstance(spec, dict):
        raise TypeError("spec musi być dict (wczytany JSON).")

    # Pobranie ustawień z pliku spec
    task = spec.get("task")
    model_kind = spec.get("model_kind")
    use_preprocessor = bool(spec.get("use_preprocessor", True))

    if task not in ("classification", "regression"):
        raise ValueError(f"Nieprawidłowy task w spec: {task}")
    if model_kind is None:
        raise ValueError("Brakuje 'model_kind' w spec.")

    # W tej prostej wersji build_model() używa globalnych RANDOM_STATE/N_JOBS.
    # Żeby wiernie odtworzyć spec, ustawiamy globalne zmienne na wartości ze spec.
    # (Najprościej i bez abstrakcji.)
    global RANDOM_STATE, N_JOBS
    if "random_state" in spec and spec["random_state"] is not None:
        RANDOM_STATE = int(spec["random_state"])
    if "n_jobs" in spec and spec["n_jobs"] is not None:
        N_JOBS = int(spec["n_jobs"])

    # Budowa pipeline tą samą ścieżką co w punkcie 9
    pipe = build_pipeline(X_sample=X_sample, task=task, model_kind=model_kind, use_preprocessor=use_preprocessor)
    return pipe


def get_features_and_target_from_spec(spec: dict) -> tuple[list[str], str]:
    """
    Zwraca listę cech i nazwę targetu z pliku spec.
    """
    feature_cols = spec.get("feature_cols")
    target_name = spec.get("target_name")

    if not isinstance(feature_cols, list) or len(feature_cols) == 0:
        raise ValueError("W spec brakuje poprawnego 'feature_cols'.")
    if not isinstance(target_name, str) or len(target_name) == 0:
        raise ValueError("W spec brakuje poprawnego 'target_name'.")

    return feature_cols, target_name


# ============================================
# 9.9 PRZEBIEG KROKU 9 (train/val/test -> target -> X/y -> pipeline -> spec)
# ============================================

from dataclasses import dataclass
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline


@dataclass
class Step9Config:
    task: str = "classification"          # "classification" | "regression"
    model_kind: str = "hgb"               # classification: "log_reg"|"rf"|"hgb" ; regression: "ridge"|"rf"|"hgb"
    close_col: str = "close"
    horizon: int = 1
    threshold: float = 0.0                # tylko dla classification
    use_preprocessor: bool = True         # True: build_preprocessor() w pipeline, False: tylko model
    artifacts_dir: Path = Path("data/modeling/build_model")
    spec_filename: str = "model_build_spec.json"
    random_state: int = 42
    n_jobs: int = -1


def run_step_9_build_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cfg: Step9Config,
    *,
    exclude_cols_extra: list[str] | None = None,
    verbose: bool = True,
):
    """
    KROK 9: train/val/test -> target -> X/y -> pipeline (bez fit) -> zapis spec.
    Zwraca dict z X/y + pipeline + spec.
    """
    # --- minimalna walidacja wejścia ---
    _check_df(train_df, "train_df")
    _check_df(val_df, "val_df")
    _check_df(test_df, "test_df")

    _check_column(train_df, cfg.close_col, "train_df")
    _check_column(val_df,   cfg.close_col, "val_df")
    _check_column(test_df,  cfg.close_col, "test_df")

    if cfg.task not in ("classification", "regression"):
        raise ValueError(f"cfg.task musi być 'classification' albo 'regression', jest: {cfg.task}")
    if not isinstance(cfg.horizon, int) or cfg.horizon <= 0:
        raise ValueError(f"cfg.horizon musi być int > 0, jest: {cfg.horizon}")

    cfg.artifacts_dir.mkdir(parents=True, exist_ok=True)

    # --- (A) target regresyjny: future log return ---
    y_reg_train = make_future_log_return(train_df, cfg.close_col, cfg.horizon, out_name=f"y_fut_lr_{cfg.horizon}d")
    y_reg_val   = make_future_log_return(val_df,   cfg.close_col, cfg.horizon, out_name=f"y_fut_lr_{cfg.horizon}d")
    y_reg_test  = make_future_log_return(test_df,  cfg.close_col, cfg.horizon, out_name=f"y_fut_lr_{cfg.horizon}d")

    # --- (B) opcjonalna klasyfikacja ---
    if cfg.task == "classification":
        y_train = make_binary_target(y_reg_train, cfg.threshold, out_name=f"y_up_{cfg.horizon}d")
        y_val   = make_binary_target(y_reg_val,   cfg.threshold, out_name=f"y_up_{cfg.horizon}d")
        y_test  = make_binary_target(y_reg_test,  cfg.threshold, out_name=f"y_up_{cfg.horizon}d")
    else:
        y_train, y_val, y_test = y_reg_train, y_reg_val, y_reg_test

    target_name = y_train.name

    # --- (C) feature cols ---
    exclude_cols = [
        "Date",
        "Open", "High", "Low", "Close", "Adj Close", "Volume",
        "open", "high", "low", "close", "adj_close", "volume",
        y_reg_train.name,   # regresyjny target
        target_name,        # finalny target (może być tym samym co y_reg)
    ]
    if exclude_cols_extra:
        exclude_cols.extend(exclude_cols_extra)

    feature_cols = infer_feature_columns(train_df, exclude_cols)

    X_train = train_df[feature_cols].copy()
    X_val   = val_df[feature_cols].copy()
    X_test  = test_df[feature_cols].copy()

    # --- (D) drop NaN w target (shift(-horizon)) ---
    X_train, y_train = drop_nan_target_rows(X_train, y_train)
    X_val,   y_val   = drop_nan_target_rows(X_val,   y_val)
    X_test,  y_test  = drop_nan_target_rows(X_test,  y_test)

    # --- (E) build pipeline (bez fit) ---
    # Uwaga: build_model() w Twoim kodzie bazuje na globalnych RANDOM_STATE/N_JOBS.
    # Żeby nie mieszać, nadpisujemy globalnie na czas budowy (prosto, bez abstrakcji).
    global RANDOM_STATE, N_JOBS
    RANDOM_STATE = int(cfg.random_state)
    N_JOBS = int(cfg.n_jobs)

    pipe: Pipeline = build_pipeline(
        X_sample=X_train,
        task=cfg.task,
        model_kind=cfg.model_kind,
        use_preprocessor=cfg.use_preprocessor
    )

    if verbose:
        print("[INFO] Shapes:")
        print("  X_train:", X_train.shape, "y_train:", y_train.shape)
        print("  X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
        print("  X_test: ", X_test.shape,  "y_test: ", y_test.shape)
        if cfg.task == "classification":
            print("[INFO] pos_rate:")
            print("  train:", float(y_train.mean()))
            print("  val:  ", float(y_val.mean()))
            print("  test: ", float(y_test.mean()))
        print("[OK] Pipeline zbudowany:")
        print(pipe)

    # --- (F) zapis spec ---
    spec = {
        "task": cfg.task,
        "model_kind": cfg.model_kind,
        "random_state": cfg.random_state,
        "n_jobs": cfg.n_jobs,
        "use_preprocessor": cfg.use_preprocessor,
        "close_col": cfg.close_col,
        "horizon": cfg.horizon,
        "threshold": cfg.threshold if cfg.task == "classification" else None,
        "feature_cols": feature_cols,
        "target_name": target_name,
    }

    spec_path = cfg.artifacts_dir / cfg.spec_filename
    save_model_build_spec(spec_path, spec)

    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val,     "y_val": y_val,
        "X_test": X_test,   "y_test": y_test,
        "pipeline": pipe,
        "spec": spec,
        "spec_path": spec_path,
        "feature_cols": feature_cols,
        "target_name": target_name,
    }

# ============================================
# 9.10 EGZEKUCJA
# ============================================


cfg = Step9Config(
    task="classification",
    model_kind="hgb",
    close_col="close",
    horizon=1,
    threshold=0.0,
    use_preprocessor=True,
    artifacts_dir=Path("data/modeling/build_model"),
    random_state=42,
    n_jobs=-1,
)

out = run_step_9_build_model(train_df, val_df, test_df, cfg, verbose=True)

model_pipe = out["pipeline"]      # gotowy pipeline do kroku 10 (fit)
spec_path  = out["spec_path"]     # zapisany JSON z konfiguracją

# jeśli chcesz wygodnie:
model_pipe = out["pipeline"]
ARTIFACTS_DIR = cfg.artifacts_dir
TASK = cfg.task

# --- eksport danych do kolejnych kroków (10/11) ---
X_train = out["X_train"]; y_train = out["y_train"]
X_val   = out["X_val"];   y_val   = out["y_val"]
X_test  = out["X_test"];  y_test  = out["y_test"]

model_pipe   = out["pipeline"]
trained_pipe = None  # będzie ustawione w kroku 10 po fit()


In [ ]:
# ============================================
# 10. Trening modelu (FIT) + zapis artefaktów (CZYSTY PUNKT 10)
# Tylko: fit() + zapis joblib (+ opcjonalnie zapis metryk do JSON)
# Wykorzystuje: X_train, y_train, X_val, y_val, model_pipe, ARTIFACTS_DIR, TASK
# ============================================

from __future__ import annotations

import json
from pathlib import Path

import joblib


class TrainingError(RuntimeError):
    """Błąd treningu modelu (punkt 10)."""


def _basic_checks(X_train, y_train, X_val, y_val):
    if X_train is None or y_train is None or X_val is None or y_val is None:
        raise TrainingError("Brakuje danych (X/y train/val).")
    if len(X_train) == 0 or len(X_val) == 0:
        raise TrainingError("X_train albo X_val jest puste.")
    if len(y_train) == 0 or len(y_val) == 0:
        raise TrainingError("y_train albo y_val jest puste.")
    if len(X_train) != len(y_train) or len(X_val) != len(y_val):
        raise TrainingError("Niezgodne długości X i y (train/val).")


def fit_and_save_artifacts(
    pipe,
    X_train, y_train,
    X_val, y_val,
    artifacts_dir: str | Path,
    task: str,
    metrics: dict | None = None,          # opcjonalnie: zapiszesz w p.10 jeśli już masz z p.11
    metrics_name: str = "metrics.json",   # nazwa pliku z metrykami jeśli metrics != None
):
    _basic_checks(X_train, y_train, X_val, y_val)

    artifacts_dir = Path(artifacts_dir)
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    # --- FIT (jedyna "logika ML" w punkcie 10) ---
    print("[INFO] fit() ...")
    pipe.fit(X_train, y_train)
    print("[OK] fit() done.")

    # --- SAVE MODEL (pipeline: preprocessing + model) ---
    model_path = artifacts_dir / "trained_model_pipeline.joblib"
    joblib.dump(pipe, model_path)
    print(f"[OK] saved model: {model_path}")

    # --- SAVE METRICS JSON [opcjonalnie] ---
    if metrics is not None:
        metrics_path = artifacts_dir / metrics_name
        metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
        print(f"[OK] saved metrics: {metrics_path}")

    return pipe


# ===== EGZEKUCJA (punkt 10) =====
try:
    # Zakładam, że te zmienne już masz po punktach 7-9:
    # X_train, y_train, X_val, y_val, model_pipe, ARTIFACTS_DIR, TASK

    trained_pipe = fit_and_save_artifacts(
        pipe=out["pipeline"],
        X_train=out["X_train"], y_train=out["y_train"],
        X_val=out["X_val"],     y_val=out["y_val"],
        artifacts_dir=cfg.artifacts_dir,
        task=cfg.task,
        metrics=None,  # metryki w punkcie 11
    )
except Exception as e:
    print("[ERROR] Punkt 10 przerwany:", repr(e))
    raise

In [ ]:
print(trained_pipe.named_steps)
print(trained_pipe.steps)
print(type(trained_pipe.named_steps["model"]))
print(trained_pipe.named_steps["model"].get_params())
print(hasattr(trained_pipe.named_steps["model"], "coef_"))
#model = trained_pipe.named_steps["model"]
#print(model.coef_)
#print(model.intercept_)

In [ ]:
# ============================================
# 11. Ewaluacja modelu
# - predykcje na val/test
# - zapis predykcji do CSV (date + FE kolumny + OHLCV bez skalowania + y_true/y_pred)
# - zapis metryk do JSON
# ============================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix
)


class EvaluationError(RuntimeError):
    """Błąd ewaluacji modelu (punkt 11)."""


def _check_inputs():
    required = [
        "trained_pipe",
        "X_val", "y_val",
        "X_test", "y_test",
        "ARTIFACTS_DIR",
        "TASK",
    ]
    missing = [k for k in required if k not in globals()]
    if missing:
        raise EvaluationError(f"[P11] Brakuje zmiennych: {missing}")


def _as_dataframe_with_date(
    X,
    fallback_df: pd.DataFrame | None,
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Zwraca DataFrame do sklejenia z predykcjami.
    Priorytet:
    1) jeśli X jest DataFrame -> użyj X
    2) else jeśli fallback_df jest DataFrame -> użyj fallback_df
    3) else -> wyjątek
    Dodatkowo tworzy kolumnę date_col jeśli jej nie ma (z DatetimeIndex lub typowych nazw).
    """
    if isinstance(X, pd.DataFrame):
        df = X.copy()
    elif isinstance(fallback_df, pd.DataFrame):
        df = fallback_df.copy()
    else:
        raise EvaluationError(
            "Nie mogę zapisać CSV z datą i oryginalnymi wartościami, bo X nie jest DataFrame "
            "i nie podano fallback_df (np. val_df/test_df)."
        )

    if date_col not in df.columns:
        if isinstance(df.index, pd.DatetimeIndex):
            df.insert(0, date_col, df.index)
        else:
            candidates = ["Date", "DATE", "datetime", "timestamp", "time"]
            found = next((c for c in candidates if c in df.columns), None)
            if found is not None:
                df = df.rename(columns={found: date_col})
            else:
                raise EvaluationError(
                    f"Brak kolumny '{date_col}' i brak DatetimeIndex — nie wiem skąd wziąć datę."
                )

    return df


def _attach_ohlcv(
    df_features: pd.DataFrame,
    raw_df: pd.DataFrame,
    ohlcv_cols: list[str],
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Dokleja kolumny OHLCV (bez skalowania) z raw_df do df_features.
    Łączy po kolumnie date_col (preferowane) lub po DatetimeIndex.
    """
    if raw_df is None or not isinstance(raw_df, pd.DataFrame):
        raise EvaluationError(
            "Brak raw_df z OHLCV. "
            "Ustaw w notebooku np. raw_df / df_raw / data_raw (DataFrame z OHLCV) "
            "z tym samym kalendarzem dat co splity."
        )

    missing = [c for c in ohlcv_cols if c not in raw_df.columns]
    if missing:
        raise EvaluationError(f"raw_df nie ma kolumn OHLCV: {missing}")

    left = df_features.copy()

    # raw -> zapewnij date_col
    if date_col not in raw_df.columns:
        if isinstance(raw_df.index, pd.DatetimeIndex):
            raw = raw_df.copy()
            raw.insert(0, date_col, raw.index)
        else:
            raise EvaluationError(f"raw_df nie ma '{date_col}' ani DatetimeIndex – nie umiem dopasować dat.")
    else:
        raw = raw_df.copy()

    # left -> zapewnij date_col
    if date_col not in left.columns:
        if isinstance(left.index, pd.DatetimeIndex):
            left.insert(0, date_col, left.index)
        else:
            raise EvaluationError(f"df_features nie ma '{date_col}' ani DatetimeIndex – nie umiem dopasować dat.")

    raw_small = raw[[date_col] + ohlcv_cols].drop_duplicates(subset=[date_col])
    out = left.merge(raw_small, on=date_col, how="left", validate="many_to_one")

    if out[ohlcv_cols].isna().any().any():
        nan_rows = int(out[ohlcv_cols].isna().any(axis=1).sum())
        raise EvaluationError(
            f"Nie udało się dopasować OHLCV dla {nan_rows} rekordów. "
            "Sprawdź, czy daty w raw_df i splitach są identyczne (ten sam timezone/format, brak przesunięć)."
        )

    return out


def _save_predictions_csv(
    *,
    split_name: str,
    X_split,
    y_true,
    y_pred,
    predictions_dir: Path,   # <- było: artifacts_path
    fallback_df: pd.DataFrame | None = None,
    date_col: str = "date",
    raw_df: pd.DataFrame | None = None,
    ohlcv_cols: list[str] | None = None,
) -> Path:
    df_base = _as_dataframe_with_date(X_split, fallback_df=fallback_df, date_col=date_col)

    out = df_base.copy()
    out["y_true"] = np.asarray(y_true)
    out["y_pred"] = np.asarray(y_pred)

    if ohlcv_cols:
        out = _attach_ohlcv(out, raw_df=raw_df, ohlcv_cols=ohlcv_cols, date_col=date_col)

    predictions_dir = Path(predictions_dir)
    predictions_dir.mkdir(parents=True, exist_ok=True)

    out_path = predictions_dir / f"{split_name}_predictions.csv"
    out.to_csv(out_path, index=False)
    return out_path


def evaluate_and_save(date_col: str = "date"):
    _check_inputs()

    artifacts_path = Path(ARTIFACTS_DIR)
    artifacts_path.mkdir(parents=True, exist_ok=True)
    predictions_dir = Path("data/modeling/evaluation")
    predictions_dir.mkdir(parents=True, exist_ok=True)

    # ============================================
    # Predykcje
    # ============================================
    try:
        y_val_pred = trained_pipe.predict(X_val)
        y_test_pred = trained_pipe.predict(X_test)
    except Exception as e:
        raise EvaluationError(f"Błąd podczas predykcji: {repr(e)}")

    # ============================================
    # Zapis predykcji do CSV (date + FE + OHLCV + y_true/y_pred)
    # ============================================
    try:
        # fallbacki, jeśli X_val/X_test nie są DataFrame
        val_fallback = globals().get("val_df", None)
        test_fallback = globals().get("test_df", None)

        # surowe dane z OHLCV (bez skalowania)
        raw_df = (
            globals().get("raw_df", None)
            or globals().get("df_raw", None)
            or globals().get("data_raw", None)
            or globals().get("df", None)
        )
        if raw_df is None:
            raise EvaluationError(
                "Nie znaleziono surowego DF z OHLCV. "
                "Dodaj w notebooku zmienną raw_df (lub df_raw / data_raw) zawierającą kolumny OHLCV."
            )

        # Dostosuj nazwy, jeśli u Ciebie OHLCV są inaczej nazwane (np. Open/High/Low/Close/Volume)
        OHLCV_COLS = ["open", "high", "low", "close", "volume"]

        p_val = _save_predictions_csv(
            split_name="val",
            X_split=X_val,
            y_true=y_val,
            y_pred=y_val_pred,
            predictions_dir=predictions_dir,
            fallback_df=val_fallback,
            date_col=date_col,
            raw_df=raw_df,
            ohlcv_cols=OHLCV_COLS,
        )

        p_test = _save_predictions_csv(
            split_name="test",
            X_split=X_test,
            y_true=y_test,
            y_pred=y_test_pred,
            predictions_dir=predictions_dir,
            fallback_df=test_fallback,
            date_col=date_col,
            raw_df=raw_df,
            ohlcv_cols=OHLCV_COLS,
        )

        print(f"[P11] Zapisano predykcje: {p_val.name}, {p_test.name}")

    except Exception as e:
        raise EvaluationError(f"Błąd zapisu predykcji z datą/FE/OHLCV: {repr(e)}")

    # ============================================
    # Metryki
    # ============================================
    metrics: dict = {}

    if TASK == "classification":
        metrics["validation"] = {
            "accuracy": float(accuracy_score(y_val, y_val_pred)),
            "precision": float(precision_score(y_val, y_val_pred, zero_division=0)),
            "recall": float(recall_score(y_val, y_val_pred, zero_division=0)),
            "f1": float(f1_score(y_val, y_val_pred, zero_division=0)),
        }
        metrics["test"] = {
            "accuracy": float(accuracy_score(y_test, y_test_pred)),
            "precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
            "recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
            "f1": float(f1_score(y_test, y_test_pred, zero_division=0)),
        }
        metrics["confusion_matrix_test"] = confusion_matrix(y_test, y_test_pred).tolist()

    elif TASK == "regression":
        metrics["validation"] = {
            "rmse": float(np.sqrt(mean_squared_error(y_val, y_val_pred))),
            "mae": float(mean_absolute_error(y_val, y_val_pred)),
            "r2": float(r2_score(y_val, y_val_pred)),
        }
        metrics["test"] = {
            "rmse": float(np.sqrt(mean_squared_error(y_test, y_test_pred))),
            "mae": float(mean_absolute_error(y_test, y_test_pred)),
            "r2": float(r2_score(y_test, y_test_pred)),
        }
    else:
        raise EvaluationError(f"Nieznany TASK: {TASK}")

    # ============================================
    # Zapis metryk
    # ============================================
    try:
        with open(artifacts_path / "metrics.json", "w") as f:
            json.dump(metrics, f, indent=4)
    except Exception as e:
        raise EvaluationError(f"Błąd zapisu metryk: {repr(e)}")

    print("\n=== METRYKI ===")
    print(json.dumps(metrics, indent=4))
    return metrics


# ===== EGZEKUCJA =====
try:
    metrics = evaluate_and_save(date_col="date")  # <- jeśli u Ciebie data ma inną nazwę, zmień tutaj
except Exception as e:
    print("[ERROR] Punkt 11 przerwany:", repr(e))
    raise


In [ ]:
# ============================================================
# 12. Regulacja modelu (RandomizedSearchCV + TimeSeriesSplit)
# - liczymy wiele metryk: F1 + Accuracy
# - wybór modelu (refit) po F1
# - zapis: best_model.joblib, best_params.json, best_score.json, cv_results.csv
# - cv_results wzbogacone o gap/stabilność/najgorszy fold (osobno dla F1 i Accuracy)
# ============================================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight


class TuningError(RuntimeError):
    """Błąd strojenia modelu (punkt 12)."""


class TuningConfig:
    """
    Konfiguracja strojenia (punkt 12) – wszystko w jednym miejscu.

    Parametry ogólne:

    n_splits:
        - liczba podziałów TimeSeriesSplit
        - więcej splitów = stabilniejsza estymacja jakości, ale większy koszt obliczeń
        - typowo 3–8 (dla danych rynkowych często 5)

    n_iter:
        - liczba losowań hiperparametrów w RandomizedSearchCV
        - więcej iteracji = większa szansa trafienia dobrego regionu, ale większy koszt
        - typowo 20–100

    scoring:
        - liczymy wiele metryk (tu: f1 oraz accuracy)
        - refit_metric określa, po której metryce wybieramy najlepszy model (refit)

    Parametry modelu (HGB):

    model__learning_rate:
        - tempo uczenia w boostingach
        - mniejsze = większy bias, mniejsza wariancja; często stabilniejsze na danych szumowych
        - typowy zakres: 0.001 – 0.2 (logarytmicznie)

    model__max_depth:
        - maksymalna głębokość drzewa bazowego
        - większa = większa złożoność, ryzyko overfittingu
        - typowy zakres: 2 – 10 (dla rynku zwykle 2–8)

    model__max_leaf_nodes:
        - maksymalna liczba liści (kontrola złożoności)
        - większa = bardziej złożony model
        - typowy zakres: 15 – 255

    model__min_samples_leaf:
        - minimalna liczba próbek w liściu
        - większa = silniejsza regularyzacja (mniejsza wariancja), możliwy underfitting
        - typowy zakres: 10 – 200

    model__l2_regularization:
        - regularyzacja L2
        - większa = mniejsza wariancja, mniejsze ryzyko overfittingu
        - typowy zakres: 1e-6 – 10 (logarytmicznie)
    """

    # Ustawienia procesu strojenia
    n_splits: int = 5
    n_iter: int = 35
    random_state: int = 42
    n_jobs: int = -1
    verbose: int = 2

    # Metryki (liczymy obie, wybieramy po f1)
    scoring = {"f1": "f1", "accuracy": "accuracy"}
    refit_metric: str = "f1"

    # Przestrzeń hiperparametrów (dla MODEL_KIND == "hgb")
    param_distributions = {
        "model__learning_rate": np.logspace(-3, -0.2, 25),
        "model__max_depth": [None, 2, 3, 5, 8],
        "model__max_leaf_nodes": [15, 31, 63, 127],
        "model__min_samples_leaf": [10, 20, 50, 100],
        "model__l2_regularization": np.logspace(-6, 1, 20),
    }

    # Wagi klas (dla HGB realizujemy przez sample_weight)
    use_balanced_sample_weight: bool = False
    # Uwaga: HGB nie ma class_weight; jeśli True, użyjemy sample_weight podczas fit().


def _check_inputs(model_pipe, X_train, y_train, X_val, y_val, task: str):
    if model_pipe is None:
        raise TuningError("model_pipe jest None (zbuduj pipeline w punkcie 9).")
    if any(x is None for x in [X_train, y_train, X_val, y_val]):
        raise TuningError("Brakuje danych (X/y train/val).")
    if task not in ("classification", "regression"):
        raise TuningError(f"Nieobsługiwany TASK: {task}")
    if len(X_train) == 0 or len(X_val) == 0:
        raise TuningError("X_train lub X_val jest puste.")
    if len(y_train) == 0 or len(y_val) == 0:
        raise TuningError("y_train lub y_val jest puste.")


def _add_diagnostics(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Dodaje kolumny diagnostyczne pod trading:
    - generalization_gap_<metric> = mean_train_<metric> - mean_test_<metric>
    - worst_fold_<metric> = min(split*_test_<metric>)
    - stability_ratio_<metric> = std_test_<metric> / mean_test_<metric>
    Działa dla multi-metric scoring (kolumny typu mean_test_f1, mean_test_accuracy...).
    """
    metrics = []
    for col in results_df.columns:
        if col.startswith("mean_test_"):
            metrics.append(col.replace("mean_test_", ""))

    for m in metrics:
        mt = f"mean_train_{m}"
        mte = f"mean_test_{m}"
        st = f"std_test_{m}"
        if mt in results_df.columns and mte in results_df.columns:
            results_df[f"generalization_gap_{m}"] = results_df[mt] - results_df[mte]

        test_cols = [c for c in results_df.columns if c.startswith("split") and c.endswith(f"_test_{m}")]
        if test_cols:
            results_df[f"worst_fold_{m}"] = results_df[test_cols].min(axis=1)

        if st in results_df.columns and mte in results_df.columns:
            results_df[f"stability_ratio_{m}"] = results_df[st] / results_df[mte].replace(0, np.nan)

    return results_df


def tune_point12(
    model_pipe,
    X_train, y_train,
    X_val, y_val,
    task: str,
    model_kind: str,
    artifacts_dir: Path,
    config: TuningConfig,
):
    _check_inputs(model_pipe, X_train, y_train, X_val, y_val, task)

    # dev = train + val (chronologicznie)
    X_dev = pd.concat([X_train, X_val], axis=0)
    y_dev = pd.concat([y_train, y_val], axis=0)

    cv = TimeSeriesSplit(n_splits=config.n_splits)

    tuning_dir = Path(artifacts_dir) / "tuning"
    tuning_dir.mkdir(parents=True, exist_ok=True)

    # Multi-metric scoring: zapisze mean_test_f1, mean_test_accuracy, itd.
    search = RandomizedSearchCV(
        estimator=model_pipe,
        param_distributions=config.param_distributions,
        n_iter=config.n_iter,
        scoring=config.scoring,
        refit=config.refit_metric,  # wybór najlepszego modelu po "f1"
        cv=cv,
        random_state=config.random_state,
        n_jobs=config.n_jobs,
        verbose=config.verbose,
        return_train_score=True,
    )

    # Opcjonalne sample_weight (przydatne dla HGB gdy klasy niezbalansowane)
    fit_kwargs = {}
    if task == "classification" and model_kind == "hgb" and config.use_balanced_sample_weight:
        sample_w = compute_sample_weight(class_weight="balanced", y=y_dev)
        fit_kwargs["model__sample_weight"] = sample_w

    try:
        search.fit(X_dev, y_dev, **fit_kwargs)
    except Exception as e:
        raise TuningError(f"RandomizedSearchCV.fit() nie powiódł się: {e}") from e

    best_pipe = search.best_estimator_
    best_params = search.best_params_

    # Dla multi-metric best_score_ to wynik dla metryki refit (tu: f1)
    best_score = float(search.best_score_)

    # cv_results + diagnostyka + zapis
    results_df = pd.DataFrame(search.cv_results_)
    results_df = _add_diagnostics(results_df)
    # Sortowanie: dla single-metric istnieje rank_test_score,
    # dla multi-metric są rank_test_<nazwa_metryki> (np. rank_test_f1)
    rank_col = "rank_test_score"
    if rank_col not in results_df.columns:
        rank_col = f"rank_test_{config.refit_metric}"  # np. rank_test_f1
        if rank_col not in results_df.columns:
            available = [c for c in results_df.columns if c.startswith("rank_test")]
            raise TuningError(f"Brakuje kolumny rangu '{rank_col}'. Dostępne: {available}")
    
    results_df = results_df.sort_values(rank_col)

    joblib.dump(best_pipe, tuning_dir / "best_model.joblib")
    (tuning_dir / "best_params.json").write_text(json.dumps(best_params, indent=2, ensure_ascii=False))
    (tuning_dir / "best_score.json").write_text(json.dumps(
        {
            "task": task,
            "model_kind": model_kind,
            "refit_metric": config.refit_metric,
            "best_cv_score": best_score,
            "n_splits": config.n_splits,
            "n_iter": config.n_iter,
        },
        indent=2,
        ensure_ascii=False
    ))
    results_df.to_csv(tuning_dir / "cv_results.csv", index=False)

    print("[OK] Strojenie zakończone.")
    print("[INFO] Best params:", best_params)
    print("[INFO] Best CV score (refit metric):", best_score)
    print("[INFO] Zapisano do:", tuning_dir)

    return best_pipe, best_params, best_score, results_df


# ======================
# EGZEKUCJA (PUNKT 12)
# Wymaga: model_pipe, X_train, y_train, X_val, y_val, ARTIFACTS_DIR, TASK, MODEL_KIND
# ======================
config = TuningConfig()

best_pipe, best_params, best_cv_score, cv_results_df = tune_point12(
    model_pipe=model_pipe,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    task=TASK,
    model_kind=MODEL_KIND,
    artifacts_dir=ARTIFACTS_DIR,
    config=config,
)

print('-----best_pipe:-----')
print(best_params)
print('-----best_cv_score:-----')
print(cv_results_df)
print('-----best_pipe:-----')
print(best_params)
print('-----best_cv_score:-----')
print(cv_results_df)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import joblib


class PredictionError(RuntimeError):
    """Błąd predykcji najlepszego modelu (po tuningu)."""


@dataclass(frozen=True)
class BestModelPredictionConfig:
    """
    Konfiguracja predykcji dla najlepszego modelu z tuningu.

    tuning_subdir:
        - podkatalog w artifacts_dir gdzie zapisany jest best_model.joblib z punktu 12

    model_filename:
        - nazwa pliku joblib z najlepszym modelem

    output_prefix:
        - prefiks nazw plików wynikowych

    include_proba:
        - czy próbować zapisać prawdopodobieństwo klasy pozytywnej (proba_pos)
        - działa tylko jeśli pipeline ma predict_proba()

    positive_class_index:
        - indeks kolumny w predict_proba dla klasy pozytywnej (typowo 1 dla {0,1})

    ohlcv_cols:
        - lista kolumn OHLCV do dołączenia z surowych danych (raw_df)
        - nazwy muszą pasować do kolumn w raw_df
    """
    tuning_subdir: str = "tuning"
    model_filename: str = "best_model.joblib"
    output_prefix: str = "best"
    include_proba: bool = True
    positive_class_index: int = 1
    ohlcv_cols: tuple[str, ...] = ("open", "high", "low", "close", "volume")


def _check_X(X):
    if X is None:
        raise PredictionError("X jest None.")
    if hasattr(X, "__len__") and len(X) == 0:
        raise PredictionError("X jest puste.")


def load_best_model(artifacts_dir: Path, config: BestModelPredictionConfig):
    """Wczytuje best_model.joblib zapisany w punkcie 12."""
    model_path = Path(artifacts_dir) / config.tuning_subdir / config.model_filename
    if not model_path.exists():
        raise PredictionError(f"Nie znaleziono modelu: {model_path}")
    return joblib.load(model_path)


def _join_ohlcv(out: pd.DataFrame, raw_df: pd.DataFrame, ohlcv_cols: tuple[str, ...]) -> pd.DataFrame:
    """
    Dołącza OHLCV z raw_df po indeksie (bez żadnego dopasowywania po dacie/kolumnach).
    Wymaga, aby raw_df miało indeks zawierający indeks out (np. DatetimeIndex).
    """
    if raw_df is None:
        return out

    missing = [c for c in ohlcv_cols if c not in raw_df.columns]
    if missing:
        raise PredictionError(f"raw_df nie zawiera wymaganych kolumn OHLCV: {missing}")

    if out.index is None:
        raise PredictionError("Brak indeksu w danych wyjściowych – nie da się bezpiecznie dołączyć OHLCV.")

    if not out.index.isin(raw_df.index).all():
        # pokazujemy kilka przykładów braków dla debugowania
        missing_idx = out.index[~out.index.isin(raw_df.index)]
        sample = list(missing_idx[:5])
        raise PredictionError(
            "Indeksy X/predykcji nie pasują do raw_df (brak części indeksów). "
            f"Przykładowe brakujące indeksy: {sample}"
        )

    ohlcv = raw_df.loc[out.index, list(ohlcv_cols)].copy()
    # kolumny OHLCV wrzucamy na początek
    return pd.concat([ohlcv, out], axis=1)


def predict_with_best_model(
    *,
    artifacts_dir: Path,
    X: pd.DataFrame,
    split_name: str,
    y_true: pd.Series | np.ndarray | None = None,
    raw_df: pd.DataFrame | None = None,
    best_pipe=None,
    config: BestModelPredictionConfig = BestModelPredictionConfig(),
) -> pd.DataFrame:
    """
    Wykonuje predykcje najlepszym modelem z punktu 12 i zapisuje do CSV.
    Dodatkowo (opcjonalnie) dołącza OHLCV z raw_df po indeksie.

    Parametry:
    - X: cechy (np. X_test). Powinno mieć indeks czasowy.
    - raw_df: surowe dane (bez skalowania) z kolumnami OHLCV i tym samym indeksem co X.
    - best_pipe: jeśli None, wczytujemy best_model.joblib z artifacts_dir/tuning.
    - y_true: opcjonalnie prawdziwe etykiety do zapisania obok predykcji.

    Zwraca:
    - DataFrame: [OHLCV...] + y_pred (+ y_true) (+ proba_pos)
    """
    _check_X(X)

    tuning_dir = Path(artifacts_dir) / config.tuning_subdir
    tuning_dir.mkdir(parents=True, exist_ok=True)

    if best_pipe is None:
        best_pipe = load_best_model(artifacts_dir=artifacts_dir, config=config)

    # Predykcje klas
    try:
        y_pred = best_pipe.predict(X)
    except Exception as e:
        raise PredictionError(f"predict() nie powiódł się: {e}") from e

    out = pd.DataFrame(index=X.index if hasattr(X, "index") else None)
    out["y_pred"] = y_pred

    # y_true (opcjonalnie)
    if y_true is not None:
        y_true_arr = y_true.values if hasattr(y_true, "values") else np.asarray(y_true)
        if len(y_true_arr) != len(out):
            raise PredictionError("Długość y_true nie pasuje do X.")
        out["y_true"] = y_true_arr

    # predict_proba (opcjonalnie) -> proba_pos
    if config.include_proba and hasattr(best_pipe, "predict_proba"):
        try:
            proba = best_pipe.predict_proba(X)
            if proba.ndim == 2 and proba.shape[1] > config.positive_class_index:
                out["proba_pos"] = proba[:, config.positive_class_index]
        except Exception:
            # proba jest opcjonalne
            pass

    # Dołączenie OHLCV (opcjonalnie)
    out = _join_ohlcv(out=out, raw_df=raw_df, ohlcv_cols=config.ohlcv_cols)

    # Zapis do CSV + metadane
    csv_path = tuning_dir / f"{config.output_prefix}_predictions_{split_name}.csv"
    out.to_csv(csv_path, index=True)

    meta = {
        "split_name": split_name,
        "n_rows": int(len(out)),
        "columns": list(out.columns),
        "has_y_true": bool(y_true is not None),
        "has_proba_pos": bool("proba_pos" in out.columns),
        "ohlcv_cols": list(config.ohlcv_cols) if raw_df is not None else [],
        "csv_path": str(csv_path),
    }
    (tuning_dir / f"{config.output_prefix}_predictions_{split_name}.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False)
    )

    print("[OK] Zapisano predykcje:", csv_path)
    return out


# ======================
# PRZYKŁADOWE UŻYCIE
# ======================
# raw_df powinien być Twoim DataFrame z oryginalnymi OHLCV i tym samym indeksem co X_*.
# np. raw_df = df[["open","high","low","close","volume"]] (plus ewentualnie inne kolumny), przed skalowaniem.

# Test:
best_test_pred_df = predict_with_best_model(
    artifacts_dir=ARTIFACTS_DIR,
    X=X_test,
    y_true=y_test,
    raw_df=df,           # <- PODSTAW swój DataFrame z OHLCV
    split_name="test",
    best_pipe=best_pipe,     # możesz pominąć, wtedy wczyta best_model.joblib
)

# Val:
best_val_pred_df = predict_with_best_model(
    artifacts_dir=ARTIFACTS_DIR,
    X=X_val,
    y_true=y_val,
    raw_df=df,
    split_name="val",
    best_pipe=best_pipe,
)

In [ ]:
# ============================================
# 13. Wdrożenie modelu (ROZDZIELONE):
#   A) build_deployment_bundle()  -> robisz RZADKO (po tuningu/retreningu)
#   B) score_with_bundle()        -> robisz CZĘSTO (na nowych / innych danych)
#
# Dopasowane do Twojego projektu (pierwszy_projekt_v4.ipynb):
# - FE: add_features_v0_1(df, output_dir, output_filename)
# - Cutoff: drop_rows_by_cutoff_date_to_csv(... cutoff_date="1962-03-14", remove="below")
# - Spec: data/modeling/build_model/model_build_spec.json (feature_cols)
# - Best model: data/modeling/tuning/best_model.joblib
# ============================================

from __future__ import annotations

import json
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import joblib


# =========================
# WYJĄTKI
# =========================

class DeploymentError(RuntimeError):
    """Błąd wdrożenia (punkt 13)."""

class DataContractError(DeploymentError):
    """Błąd kontraktu danych (RAW/FE)."""

class ArtifactError(DeploymentError):
    """Błąd operacji na artefaktach (plikach)."""


# =========================
# KONFIG
# =========================

@dataclass(frozen=True)
class DeploymentConfig:
    """
    Konfiguracja wdrożenia (punkt 13) — pod batch scoring dla danych rynkowych.

    Parametry
    ----------
    best_model_src_path : str
        Ścieżka do najlepszego modelu po tuningu (źródło).
        Wpływ na bias/variance: brak (to wybór finalnego artefaktu).

    build_spec_src_path : str
        Ścieżka do model_build_spec.json (źródło), zawiera m.in. feature_cols.
        Wpływ: brak, ale krytyczne dla spójności cech.

    deployment_dir : str
        Katalog docelowy na bundle produkcyjny (model + kontrakt + metadata).
        Wpływ: brak; organizacja.

    deployment_model_name : str
        Nazwa pliku modelu w bundle.
        Wpływ: brak.

    feature_cols_name : str
        Nazwa pliku z kontraktem cech w bundle.
        Wpływ: brak.

    metadata_name : str
        Nazwa pliku z metadanymi w bundle.
        Wpływ: brak.

    required_ohlcv_cols : tuple[str, ...]
        Minimalny kontrakt wejścia dla RAW OHLCV.
        Wpływ: brak; walidacja.

    cutoff_date : str
        Data odcięcia jak w notebooku (u Ciebie 1962-03-14).
        Wpływ: pośredni (zmienia zbiór), ale to element czyszczenia.

    cutoff_remove : str
        "below" usuwa daty < cutoff (zostają >=).
        Wpływ: jw.

    positive_class_index : int
        Którą kolumnę z predict_proba traktujesz jako proba_pos.
        Wpływ: brak na bias/variance; wpływa na interpretację.

    fe_output_dir : str
        Gdzie FE zapisuje pliki cech (log/artefakt).
        Wpływ: brak.

    validation_out_dir : str
        Gdzie helper do cutoff zapisuje CSV z walidacji.
        Wpływ: brak.
    """
    best_model_src_path: str = "data/modeling/build_model/tuning/best_model.joblib"
    build_spec_src_path: str = "data/modeling/build_model/model_build_spec.json"

    deployment_dir: str = "data/modeling/deployment"
    deployment_model_name: str = "model.joblib"
    feature_cols_name: str = "feature_cols.json"
    metadata_name: str = "metadata.json"
    build_spec_copy_name: str = "model_build_spec.json"

    required_ohlcv_cols: Tuple[str, ...] = ("open", "high", "low", "close", "volume")

    cutoff_date: str = "1962-03-14"
    cutoff_remove: str = "below"

    positive_class_index: int = 1

    fe_output_dir: str = "data/feature_engineering/deployment"
    validation_out_dir: str = "data/validation"


CFG = DeploymentConfig()


# =========================
# IO / HELPERS
# =========================

def _ensure_datetime_index(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Zapewnia DatetimeIndex i monotoniczność (istotne dla TS).
    """
    df = df_in.copy()
    if isinstance(df.index, pd.DatetimeIndex):
        if not df.index.is_monotonic_increasing:
            df = df.sort_index()
        return df

    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="raise")
        df = df.sort_values("Date").set_index("Date", drop=True)
        return df

    raise DataContractError("Dane nie mają DatetimeIndex ani kolumny 'Date'.")


def _load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise ArtifactError(f"Brak pliku: {path}")
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        raise ArtifactError(f"Nie mogę wczytać JSON: {path}. Błąd: {repr(e)}")


def _save_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _copy_file(src: Path, dst: Path) -> None:
    if not src.exists():
        raise ArtifactError(f"Brak pliku źródłowego: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(src, dst)
    except Exception as e:
        raise ArtifactError(f"Nie mogę skopiować {src} -> {dst}. Błąd: {repr(e)}")

import hashlib

def _sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """
    Liczy SHA256 pliku w sposób strumieniowy (bez ładowania całego do RAM).

    Parametry
    ----------
    path : Path
        Ścieżka do pliku.
    chunk_size : int
        Rozmiar chunku w bajtach. Większy = szybciej, ale więcej RAM.
        Typowo: 1–8 MB.

    Wpływ na bias/variance: brak (to tylko kontrola wersji artefaktu).
    """
    if not path.exists():
        raise ArtifactError(f"Nie znaleziono pliku do hashowania: {path}")

    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _load_model(path: Path):
    if not path.exists():
        raise ArtifactError(f"Brak modelu: {path}")
    try:
        return joblib.load(path)
    except Exception as e:
        raise ArtifactError(f"Nie mogę wczytać modelu joblib: {path}. Błąd: {repr(e)}")


def _validate_raw_ohlcv(df_raw: pd.DataFrame, required_cols: Tuple[str, ...]) -> None:
    if df_raw is None or not isinstance(df_raw, pd.DataFrame) or df_raw.empty:
        raise DataContractError("RAW OHLCV jest puste lub nie jest DataFrame.")
    missing = [c for c in required_cols if c not in df_raw.columns]
    if missing:
        raise DataContractError(f"Brakuje wymaganych kolumn OHLCV: {missing}")


def _require_callable(name: str) -> None:
    if name not in globals() or not callable(globals()[name]):
        raise DeploymentError(f"Brak funkcji '{name}' w globals() lub nie jest callable. Uruchom odpowiednią komórkę.")


# =========================
# A) BUNDLE (RZADKO)
# =========================

def build_deployment_bundle(cfg: DeploymentConfig = CFG) -> Path:
    """
    Buduje (lub odświeża) bundle produkcyjny:
    - kopiuje best model -> deployment/model.joblib
    - kopiuje model_build_spec.json
    - zapisuje feature_cols.json (kontrakt wejścia)
    - zapisuje metadata.json

    Kiedy uruchamiać:
    - po zakończonym tuningu / retreningu,
    - gdy zmienisz FE lub zestaw cech (feature_cols).

    Zwraca:
    - Path do katalogu bundle.
    """
    dep_dir = Path(cfg.deployment_dir)
    dep_dir.mkdir(parents=True, exist_ok=True)

    src_model = Path(cfg.best_model_src_path)
    src_spec = Path(cfg.build_spec_src_path)

    # (1) Kopie źródeł do bundle
    _copy_file(src_model, dep_dir / cfg.deployment_model_name)
    _copy_file(src_spec, dep_dir / cfg.build_spec_copy_name)

    # (2) Feature cols z "spec"
    spec = _load_json(src_spec)
    feature_cols = spec.get("feature_cols", None)
    if not isinstance(feature_cols, list) or len(feature_cols) == 0:
        raise ArtifactError(f"Spec nie zawiera poprawnego 'feature_cols': {src_spec}")

    _save_json(dep_dir / cfg.feature_cols_name, {"feature_cols": feature_cols})

    # (3) Metadane
    src_model_sha256 = _sha256_file(src_model)

    metadata = {
        "created_at_utc": datetime.utcnow().isoformat() + "Z",
        "source_best_model": cfg.best_model_src_path,
        "source_build_spec": cfg.build_spec_src_path,
        "source_best_model_sha256": src_model_sha256,  # <- NOWE
        "required_ohlcv_cols": list(cfg.required_ohlcv_cols),
        "cutoff_date": cfg.cutoff_date,
        "cutoff_remove": cfg.cutoff_remove,
        "positive_class_index": cfg.positive_class_index,
        "n_feature_cols": int(len(feature_cols)),
    }
    _save_json(dep_dir / cfg.metadata_name, metadata)

    print("✅ Bundle wdrożeniowy gotowy.")
    print(f" - {dep_dir.resolve()}")
    print(f" - Model: {cfg.deployment_model_name}")
    print(f" - Kontrakt cech: {cfg.feature_cols_name}")

    return dep_dir


# =========================
# B) SCORING (CZĘSTO)
# =========================

def score_with_bundle(
    df_raw_ohlcv: pd.DataFrame,
    bundle_dir: str | Path = CFG.deployment_dir,
    out_csv_path: Optional[str | Path] = None,
    cfg: DeploymentConfig = CFG,
) -> pd.DataFrame:
    """
    Scoring na nowych / innych danych z użyciem istniejącego bundle.

    Wejście:
    - df_raw_ohlcv: surowe OHLCV (DatetimeIndex lub kolumna 'Date')
    - bundle_dir: katalog z model.joblib + feature_cols.json
    - out_csv_path: opcjonalny zapis predykcji

    Wykonuje:
    - walidację kontraktu OHLCV
    - FE: add_features_v0_1
    - cutoff: drop_rows_by_cutoff_date_to_csv (jak w notebooku)
    - wybór feature_cols z bundle
    - predict + predict_proba (jeśli dostępne)
    - zwraca DataFrame: OHLCV + y_pred + proba_pos

    Zwraca:
    - DataFrame z wynikami scoringu.
    """
    # wymagamy Twoich funkcji z notebooka
    _require_callable("add_features_v0_1")
    _require_callable("drop_rows_by_cutoff_date_to_csv")

    # (1) kontrakt RAW
    df_raw = _ensure_datetime_index(df_raw_ohlcv)
    _validate_raw_ohlcv(df_raw, cfg.required_ohlcv_cols)

    # (2) FE (zapis pliku FE jako log)
    fe_filename = f"features_score_{datetime.utcnow().strftime('%Y-%m-%dT%H%M%SZ')}.csv"
    df_feat = add_features_v0_1(df=df_raw, output_dir=cfg.fe_output_dir, output_filename=fe_filename)

    # (3) cutoff (jak w notebooku)
    df_feat_cleaned = drop_rows_by_cutoff_date_to_csv(
        df_feat,
        date_col="Date",
        cutoff_date=cfg.cutoff_date,
        remove=cfg.cutoff_remove,
        out_dir=cfg.validation_out_dir,
    )
    df_feat_cleaned = _ensure_datetime_index(df_feat_cleaned)

    # (4) wczytanie bundle
    bundle_dir = Path(bundle_dir)
    model_path = bundle_dir / cfg.deployment_model_name
    cols_path = bundle_dir / cfg.feature_cols_name

    model = _load_model(model_path)
    feature_cols_obj = _load_json(cols_path)
    feature_cols = feature_cols_obj.get("feature_cols", None)
    if not isinstance(feature_cols, list) or len(feature_cols) == 0:
        raise ArtifactError(f"Bundle ma niepoprawny {cfg.feature_cols_name}: {cols_path}")

    # (5) X
    missing_feat = [c for c in feature_cols if c not in df_feat_cleaned.columns]
    if missing_feat:
        raise DataContractError(f"Po FE brakuje kolumn z kontraktu cech bundle: {missing_feat}")

    X = df_feat_cleaned[feature_cols].copy()

    # bezpiecznik: całe kolumny NaN zwykle oznaczają, że FE nie ma historii/okna
    all_nan_cols = [c for c in X.columns if X[c].isna().all()]
    if all_nan_cols:
        raise DataContractError(f"Kolumny cech w 100% NaN (za mało historii / problem FE): {all_nan_cols[:20]}")

    # (6) predict
    try:
        y_pred = model.predict(X)
    except Exception as e:
        raise DeploymentError(f"model.predict() nie zadziałał. Błąd: {repr(e)}")

    proba_pos = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X)
            if proba.ndim == 2 and proba.shape[1] > cfg.positive_class_index:
                proba_pos = proba[:, cfg.positive_class_index]
        except Exception:
            proba_pos = None

    # (7) wynik (samowystarczalny format do integracji)
    out = pd.DataFrame(index=X.index)
    out.index.name = "Date"
    for c in cfg.required_ohlcv_cols:
        out[c] = df_raw.reindex(out.index)[c]

    out["y_pred"] = y_pred
    if proba_pos is not None:
        out["proba_pos"] = proba_pos

    # (8) zapis opcjonalny
    if out_csv_path is not None:
        out_csv_path = Path(out_csv_path)
        out_csv_path.parent.mkdir(parents=True, exist_ok=True)
        out.to_csv(out_csv_path, index=True)
        print(f"[INFO] Zapisano scoring do: {out_csv_path.resolve()}")

    print("✅ Scoring zakończony.")
    print(f" - N: {len(out)} | proba_pos: {'TAK' if proba_pos is not None else 'NIE'}")
    return out


# =========================
# EGZEKUCJA (PRZYKŁAD)
# =========================
# 1) RAZ: budujesz bundle po tuningu / wyborze modelu:
# dep_dir = build_deployment_bundle(CFG)
#
# 2) CZĘSTO: scoring na nowych / innych danych:
# new_preds = score_with_bundle(df_raw_ohlcv=df, bundle_dir=dep_dir, out_csv_path="data/modeling/deployment/batch_predictions_new.csv")

try:
    dep_dir = Path(CFG.deployment_dir)
    dep_model = dep_dir / CFG.deployment_model_name
    dep_cols = dep_dir / CFG.feature_cols_name
    dep_meta = dep_dir / CFG.metadata_name

    src_model = Path(CFG.best_model_src_path)

    bundle_missing = (not dep_model.exists()) or (not dep_cols.exists()) or (not dep_meta.exists())

    # Heurystyka mtime (szybka), ale decyzję finalną robi hash
    bundle_stale_mtime = False
    if (not bundle_missing) and src_model.exists():
        bundle_stale_mtime = src_model.stat().st_mtime > dep_model.stat().st_mtime

    bundle_stale_hash = False
    if (not bundle_missing) and src_model.exists():
        meta = _load_json(dep_meta)
        old_sha = meta.get("source_best_model_sha256", None)
        new_sha = _sha256_file(src_model)

        # jeśli meta nie ma hasha (np. stare bundle), traktujemy jako stale
        if (old_sha is None) or (old_sha != new_sha):
            bundle_stale_hash = True

    if bundle_missing:
        print("[INFO] Bundle nie istnieje -> buduję bundle.")
        build_deployment_bundle(CFG)
    elif bundle_stale_hash:
        print("[INFO] Wykryto zmianę modelu (SHA256) -> redeploy bundle.")
        build_deployment_bundle(CFG)
    elif bundle_stale_mtime:
        # opcjonalnie: mtime może się zmienić przy kopiowaniu, ale jeśli hash zgodny to ignorujemy
        print("[INFO] Model źródłowy nowszy (mtime), ale hash zgodny -> pomijam redeploy, robię tylko scoring.")
    else:
        print("[INFO] Bundle aktualny -> pomijam budowę, robię tylko scoring.")

    # --- Scoring na aktualnym df (RAW) ---
    df_raw_for_score = globals().get("df", None)
    if df_raw_for_score is None:
        raise DataContractError("Nie znaleziono globalnej zmiennej 'df' z RAW OHLCV.")

    preds = score_with_bundle(
        df_raw_ohlcv=df_raw_for_score,
        bundle_dir=CFG.deployment_dir,
        out_csv_path="data/modeling/deployment/batch_predictions.csv",
        cfg=CFG,
    )
    display(preds.head(10))

except Exception as e:
    print("[ERROR] Punkt 13 przerwany:", repr(e))
    raise